In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

%load_ext autoreload
%reload_ext autoreload
%autoreload 2
# this is important to be able to re-import the module after making modifications to the zavolab_pyutils code on Scicore

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return "ns"
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return "ns"

In [ ]:
# 1. Load the environment variables
load_dotenv("pbody_RNA_Spang.scicore.env",override=True)

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "celegans_annotation_dir": os.getenv("CELEGANS_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "configs_dir": os.getenv("CONFIGS_DIR"),
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "genome_file": os.getenv("YEAST_GENOME_FILE"),
    "annotation_file": os.getenv("YEAST_ANNOTATION_FILE"),
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Make custom annotations

https://academic.oup.com/nar/article/48/D1/D743/5587629

https://link.springer.com/article/10.1007/s00438-014-0913-6

## Extract start codons from basic GTF

### v1

In [ ]:
# main yeast gtf
gtf_df = pd.read_csv(
    file_paths["annotation_file"],
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=5,
)

In [ ]:
WF_version = "v1"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/modified_annotation/"
command = "mkdir -p " + dir_path
out = subprocess.check_output(command, shell=True)

start_codons_df = gtf_df.loc[gtf_df[2] == "start_codon"].reset_index(drop=True)
start_codons_df["gene_id"] = (
    start_codons_df[8]
    .str.split('gene_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
start_codons_df["transcript_id"] = (
    start_codons_df[8]
    .str.split('transcript_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
start_codons_bed = (
    start_codons_df[[0, 3, 4, "gene_id", 7, 6]].drop_duplicates().reset_index(drop=True)
)
print(len(start_codons_bed))
start_codons_bed[7] = 0
start_codons_bed["gene_id"] = start_codons_bed.groupby([0, 3, 4, 7, 6])[
    "gene_id"
].transform(lambda x: "___".join(x))
start_codons_bed = start_codons_bed.drop_duplicates().reset_index(drop=True)
start_codons_bed[3] = start_codons_bed[3] - 1
print(len(start_codons_bed))
start_codons_bed.to_csv(
    dir_path + "start_codons.bed",
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

command = (
    "bedtools sort -i "
    + dir_path
    + "start_codons.bed"
    + " > "
    + dir_path
    + "start_codons.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

In [ ]:
dir_path + "start_codons.sorted.bed"

## Append UTRs to basic GTF

### v1

In [ ]:
# main yeast gtf
gtf_df = pd.read_csv(
    file_paths["annotation_file"],
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=5,
)

# TIF-seq data, indicating transcript ends
tif_seq_gal = pd.read_csv(
    subdirs["annotation_dir"] + "longest_full-ORF_transcripts_gal.gff3",
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=3,
)
tif_seq_ypd = pd.read_csv(
    subdirs["annotation_dir"] + "longest_full-ORF_transcripts_ypd.gff3",
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=3,
)

tif_seq_gal["gene_id"] = (
    tif_seq_gal[8].str.split(";ID=", expand=True)[1].str.split("_", expand=True)[0]
)
tif_seq_ypd["gene_id"] = (
    tif_seq_ypd[8].str.split(";ID=", expand=True)[1].str.split("_", expand=True)[0]
)

tif_seq = pd.merge(
    tif_seq_ypd.rename(columns={3: "start_ypd", 4: "end_ypd"})[
        ["gene_id", 0, 6, "start_ypd", "end_ypd"]
    ],
    tif_seq_gal.rename(columns={3: "start_gal", 4: "end_gal"})[
        ["gene_id", 0, 6, "start_gal", "end_gal"]
    ],
    how="outer",
    on=["gene_id", 0, 6],
)
tif_seq["start"] = (
    tif_seq[["start_ypd", "start_gal"]].min(1).astype("int") - 1
)  # bed format
tif_seq["end"] = tif_seq[["end_ypd", "end_gal"]].max(1).astype("int")

tif_seq["score"] = 0
tif_seq = tif_seq[[0, "start", "end", "gene_id", "score", 6]]
tif_seq[0] = tif_seq[0].str.replace("chr", "")

tif_seq.to_csv(
    subdirs["temp_dir"] + "tif_seq.bed", sep=str("\t"), header=False, index=None
)
command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + "tif_seq.bed > "
    + subdirs["temp_dir"]
    + "tif_seq.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

gtf_bed = gtf_df.loc[(gtf_df[2] == "CDS")].copy().reset_index(drop=True)
# in SGD GTF file, CDS does not include stop codon, but we want to include it
gtf_bed[3] = gtf_bed.apply(lambda x: x[3] - 3 if x[6] == "-" else x[3], 1)
gtf_bed[4] = gtf_bed.apply(lambda x: x[4] + 3 if x[6] == "+" else x[4], 1)

gtf_bed["transcript_id"] = (
    gtf_bed[8]
    .str.split('transcript_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
gtf_bed[3] = gtf_bed[3] - 1  # bed format
print(len(gtf_bed))
gtf_bed = (
    gtf_bed.groupby([0, 6, "transcript_id"]).agg({3: min, 4: max}).reset_index()
)  # ensure that CDS is defined as the most distal borders of exonic CDSs
print(len(gtf_bed))
gtf_bed["score"] = 0
gtf_bed[[0, 3, 4, "transcript_id", "score", 6]].to_csv(
    subdirs["temp_dir"] + "basic_gtf.bed", sep=str("\t"), header=False, index=None
)
command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + "basic_gtf.bed > "
    + subdirs["temp_dir"]
    + "basic_gtf.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

command = (
    "bedtools intersect -sorted -wao -s -a "
    + subdirs["temp_dir"]
    + "basic_gtf.sorted.bed -b "
    + subdirs["temp_dir"]
    + "tif_seq.sorted.bed > "
    + subdirs["temp_dir"]
    + "intersection.bed"
)
out = subprocess.check_output(command, shell=True)

intersection = pd.read_csv(
    subdirs["temp_dir"] + "intersection.bed",
    delimiter="\t",
    index_col=None,
    header=None,
)
intersection["host_gene"] = intersection[3].str.split("_mRNA", expand=True)[0]

In [ ]:
len(
    intersection.loc[
        (intersection["host_gene"] != intersection[9]) & (intersection[9] != ".")
    ]
), len(intersection.loc[(intersection[9] != ".")])

In [ ]:
# there are intersections with other genes, but in all these cases, there is also intersection with the correct gene
l = list(
    intersection.loc[
        (intersection["host_gene"] != intersection[9]) & (intersection[9] != ".")
    ][9].unique()
)
len(
    intersection.loc[
        (intersection["host_gene"] == intersection[9]) & (intersection[9].isin(l))
    ][9].unique()
), len(l)

In [ ]:
intersection = intersection.loc[
    (intersection["host_gene"] == intersection[9]) | (intersection[9] == ".")
].reset_index(drop=True)

In [ ]:
len(intersection.drop_duplicates([3])), len(
    intersection
)  # there is only single intersection for each CDS

In [ ]:
# now make a custom GTF file with only one element: exon
# at that, create artificial transcript ids, corresponding to CDS, 5'UTR, and 3'UTR

CDS_gtf = gtf_df.loc[(gtf_df[2] == "CDS")].copy().reset_index(drop=True)
CDS_gtf[2] = "exon"
# in SGD GTF file, CDS does not include stop codon, but we want to include it
CDS_gtf[3] = CDS_gtf.apply(lambda x: x[3] - 3 if x[6] == "-" else x[3], 1)
CDS_gtf[4] = CDS_gtf.apply(lambda x: x[4] + 3 if x[6] == "+" else x[4], 1)

CDS_gtf["transcript_id"] = (
    CDS_gtf[8]
    .str.split('transcript_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
CDS_gtf["new_transcript_id"] = CDS_gtf["transcript_id"] + "___CDS"
CDS_gtf["8_new"] = CDS_gtf.apply(
    lambda x: x[8].split('transcript_id "')[0]
    + 'transcript_id "'
    + x["new_transcript_id"]
    + '"'
    + '"'.join(x[8].split('transcript_id "')[1].split('"')[1:]),
    1,
)

# now extract UTRs
tmp1 = intersection.loc[(intersection[9] != ".") & (intersection[7] < intersection[1])][
    [0, 7, 1, 3, 5]
].reset_index(drop=True)
tmp1["start"] = tmp1[7] + 1  # back to gtf format
tmp1["end"] = tmp1[1]  # back to gtf format, but subtract 1 bp
tmp1 = tmp1.rename(columns={0: "chr", 3: "transcript_id", 5: "strand"})
tmp1["type"] = tmp1.apply(lambda x: "5UTR" if x["strand"] == "+" else "3UTR", 1)
tmp1 = tmp1[["chr", "start", "end", "transcript_id", "strand", "type"]]

tmp2 = intersection.loc[(intersection[9] != ".") & (intersection[8] > intersection[2])][
    [0, 2, 8, 3, 5]
].reset_index(drop=True)
tmp2["start"] = tmp2[2] + 1
tmp2["end"] = tmp2[8]
tmp2 = tmp2.rename(columns={0: "chr", 3: "transcript_id", 5: "strand"})
tmp2["type"] = tmp2.apply(lambda x: "3UTR" if x["strand"] == "+" else "5UTR", 1)
tmp2 = tmp2[["chr", "start", "end", "transcript_id", "strand", "type"]]

utr_gtf = pd.concat([tmp1, tmp2]).reset_index(drop=True)
utr_gtf["new_transcript_id"] = utr_gtf["transcript_id"] + "___" + utr_gtf["type"]

utr_gtf = pd.merge(
    utr_gtf,
    CDS_gtf[["transcript_id", 8]].drop_duplicates(["transcript_id"]),
    how="left",
    on="transcript_id",
)


def get_8_new(x):
    # first, we substitute old transcript id with new transcript id
    new_8 = (
        x[8].split('transcript_id "')[0]
        + 'transcript_id "'
        + x["new_transcript_id"]
        + '"'
        + '"'.join(x[8].split('transcript_id "')[1].split('"')[1:])
    )
    # second, we make sure that exon number is "1"
    new_8 = (
        new_8.split('exon_number "')[0]
        + 'exon_number "1"'
        + '"'.join(x[8].split('exon_number "')[1].split('"')[1:])
    )
    return new_8


utr_gtf["8_new"] = utr_gtf.apply(lambda x: get_8_new(x), 1)
utr_gtf[0] = utr_gtf["chr"]
utr_gtf[1] = "TIFseq"
utr_gtf[2] = "exon"
utr_gtf[3] = utr_gtf["start"]
utr_gtf[4] = utr_gtf["end"]
utr_gtf[5] = "."
utr_gtf[6] = utr_gtf["strand"]
utr_gtf[7] = 0

modified_gtf = (
    pd.concat(
        [
            CDS_gtf[list(range(0, 8)) + ["8_new", "transcript_id"]],
            utr_gtf[list(range(0, 8)) + ["8_new", "transcript_id"]],
        ]
    )
    .sort_values("transcript_id")
    .reset_index(drop=True)
)

In [ ]:
subdirs["temp_dir"] + "genome_fai.sorted.bed"

In [ ]:
# now, fill the gaps between annotated elements
# we will go with assumption that if the region is annotated as protein-coding (currently), we fully trust it.
# so, if it overlaps with some other non-coding region, we will still interpret it as coding

all_exons_bed = gtf_df.loc[gtf_df[2] == "exon"].copy().reset_index(drop=True)
all_exons_bed["transcript_id"] = (
    all_exons_bed[8]
    .str.split('transcript_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
all_exons_bed["exon_number"] = (
    all_exons_bed[8]
    .str.split('exon_number "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
all_exons_bed["name"] = (
    all_exons_bed["transcript_id"] + "___" + all_exons_bed["exon_number"]
)
all_exons_bed["gene_biotype"] = (
    all_exons_bed[8]
    .str.split('gene_biotype "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
all_exons_bed["score"] = 0
all_exons_bed[3] = all_exons_bed[3] - 1
all_exons_bed[[0, 3, 4, "name", "score", 6]].to_csv(
    subdirs["temp_dir"] + "all_exons_gtf.bed", sep=str("\t"), header=False, index=None
)

command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + "all_exons_gtf.bed > "
    + subdirs["temp_dir"]
    + "all_exons_gtf.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

prot_coding_bed = modified_gtf[[0, 3, 4, "transcript_id", 7, 6]].copy()
prot_coding_bed[3] = prot_coding_bed[3] - 1
prot_coding_bed.to_csv(
    subdirs["temp_dir"] + "prot_coding.bed", sep=str("\t"), header=False, index=None
)

command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + """prot_coding.bed | bedtools merge -s -i stdin -c 6 -o distinct | \
awk -F'\t' 'OFS="\t" {print $1, $2, $3, "temp", "0", $4}' | bedtools sort -i stdin > """
    + subdirs["temp_dir"]
    + "prot_coding.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

command = (
    "bedtools subtract -s -a "
    + subdirs["temp_dir"]
    + "all_exons_gtf.sorted.bed -b "
    + subdirs["temp_dir"]
    + "prot_coding.sorted.bed > "
    + subdirs["temp_dir"]
    + "all_exons_gtf.remaining.bed"
)
out = subprocess.check_output(command, shell=True)

all_exons_remaining = pd.read_csv(
    subdirs["temp_dir"] + "all_exons_gtf.remaining.bed",
    delimiter="\t",
    index_col=None,
    header=None,
)

prot_coding_bed.columns = list(range(0, 6))
all_annoted_elements_bed = pd.concat(
    [all_exons_remaining, prot_coding_bed]
).reset_index(drop=True)

# all the rest will be intergenic
genome_fai_bed = pd.read_csv(
    file_paths["genome_file_fai"], delimiter="\t", index_col=None, header=None
)
genome_fai_bed["chr"] = genome_fai_bed[0]
genome_fai_bed["start"] = 0
genome_fai_bed["end"] = genome_fai_bed[1]
genome_fai_bed["transcript_id"] = "intergenic"
genome_fai_bed["score"] = 0
genome_fai_bed["strand"] = "+"
tmp1 = genome_fai_bed[
    ["chr", "start", "end", "transcript_id", "score", "strand"]
].copy()
tmp1["strand"] = "-"
genome_fai_bed = pd.concat(
    [genome_fai_bed[["chr", "start", "end", "transcript_id", "score", "strand"]], tmp1]
).reset_index(drop=True)

genome_fai_bed.to_csv(
    subdirs["temp_dir"] + "genome_fai.bed", sep=str("\t"), header=False, index=None
)
command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + "genome_fai.bed > "
    + subdirs["temp_dir"]
    + "genome_fai.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

all_annoted_elements_bed.to_csv(
    subdirs["temp_dir"] + "all_annoted_elements.bed",
    sep=str("\t"),
    header=False,
    index=None,
)
command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + """all_annoted_elements.bed | bedtools merge -s -i stdin -c 6 -o distinct | \
awk -F'\t' 'OFS="\t" {print $1, $2, $3, "temp", "0", $4}' | bedtools sort -i stdin > """
    + subdirs["temp_dir"]
    + "all_annoted_elements.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

command = (
    "bedtools subtract -s -a "
    + subdirs["temp_dir"]
    + "genome_fai.sorted.bed -b "
    + subdirs["temp_dir"]
    + "all_annoted_elements.sorted.bed > "
    + subdirs["temp_dir"]
    + "intergenic.bed"
)
out = subprocess.check_output(command, shell=True)

intergenic = pd.read_csv(
    subdirs["temp_dir"] + "intergenic.bed", delimiter="\t", index_col=None, header=None
)

# now, convert bed format to gtf format

all_exons_remaining_gtf = all_exons_remaining.copy()
all_exons_remaining_gtf[1] = all_exons_remaining_gtf[1] + 1  # from bed to gtf format
all_exons_remaining_gtf["len"] = (
    all_exons_remaining_gtf[2] - all_exons_remaining_gtf[1] + 1
)
all_exons_remaining_gtf = all_exons_remaining_gtf.loc[
    all_exons_remaining_gtf["len"] > 0
].reset_index(drop=True)
all_exons_remaining_gtf["transcript_id"] = all_exons_remaining_gtf[3].str.split(
    "___", expand=True
)[0]
all_exons_remaining_gtf["exon_number"] = all_exons_remaining_gtf[3].str.split(
    "___", expand=True
)[1]
all_exons_remaining_gtf = pd.merge(
    all_exons_remaining_gtf,
    all_exons_bed[["transcript_id", "exon_number", 8]],
    how="left",
    on=["transcript_id", "exon_number"],
)

all_exons_remaining_gtf = all_exons_remaining_gtf.rename(
    columns={1: 3, 2: 4, 5: 6, 3: "tmp1", 4: 7, 8: "8_new"}
).drop(["tmp1"], axis=1)
all_exons_remaining_gtf[2] = "exon"
all_exons_remaining_gtf[1] = "sgd_modified"
all_exons_remaining_gtf[5] = "."
all_exons_remaining_gtf = all_exons_remaining_gtf.sort_values(
    ["transcript_id", "exon_number"]
).reset_index(drop=True)[list(range(0, 8)) + ["8_new", "transcript_id"]]

intergenic_gtf = intergenic.copy()
intergenic_gtf["transcript_id"] = "intergenic_" + (intergenic_gtf.index).astype("str")
intergenic_gtf[1] = intergenic_gtf[1] + 1  # from bed to gtf format
intergenic_gtf["8_new"] = (
    'gene_id "'
    + intergenic_gtf["transcript_id"]
    + '"; transcript_id "'
    + intergenic_gtf["transcript_id"]
    + '"; exon_number "1"; gene_source "genome"; gene_biotype "intergenic"; transcript_source "genome"; transcript_biotype "intergenic"; protein_id "'
    + intergenic_gtf["transcript_id"]
    + '"; tag "genome";'
)
intergenic_gtf = intergenic_gtf.rename(columns={1: 3, 2: 4, 5: 6, 4: 7, 3: "tmp1"})
intergenic_gtf[2] = "exon"
intergenic_gtf[1] = "genome"
intergenic_gtf[5] = "."

intergenic_gtf = intergenic_gtf[list(range(0, 8)) + ["8_new", "transcript_id"]]

final_modified_gtf = (
    pd.concat([intergenic_gtf, all_exons_remaining_gtf, modified_gtf])
    .reset_index(drop=True)
    .drop(["transcript_id"], axis=1)
)

In [ ]:
WF_version = "v1"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/modified_annotation/"
command = "mkdir -p " + dir_path
out = subprocess.check_output(command, shell=True)

final_modified_gtf.to_csv(
    dir_path + "CDS_and_UTRs.gtf",
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

In [ ]:
dir_path + "CDS_and_UTRs.gtf"

### Draft

In [ ]:
# main yeast gtf
gtf_df = pd.read_csv(
    file_paths["annotation_file"],
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=5,
)

# RNA central
rna_central = pd.read_csv(
    subdirs["annotation_dir"] + "rna_central_saccharomyces_cerevisiae.R64-1-1.gff3",
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=1,
)
rna_central[2] = rna_central[2].str.replace("noncoding_exon", "exon")
rna_central["gene_biotype"] = rna_central[8].str.split(";type=|;", expand=True)[1]
rna_central["gene_source"] = "RNA_central"
rna_central["gene_id"] = rna_central[8].str.split("ID=|;|:", expand=True)[4]

rna_central_exons = (
    rna_central.loc[rna_central[2] == "exon"].copy().reset_index(drop=True)
)
rna_central_exons["exon_id"] = (
    rna_central_exons["gene_id"]
    + ".transcript"
    + "."
    + rna_central_exons[8].str.split("ID=|;|:", expand=True)[5]
)
rna_central_exons["exon_number"] = (
    rna_central_exons[8]
    .str.split("ID=|;|:", expand=True)[5]
    .str.split("exon", expand=True)
    .iloc[:, -1]
)
rna_central_exons[8] = (
    'gene_id "'
    + rna_central_exons["gene_id"]
    + '"; transcript_id "'
    + rna_central_exons["gene_id"]
    + ".transcript"
    + '"; exon_number "'
    + rna_central_exons["exon_number"]
    + '"; gene_source "'
    + rna_central_exons["gene_source"]
    + '"; gene_biotype "'
    + rna_central_exons["gene_biotype"]
    + '"; transcript_source "'
    + rna_central_exons["gene_source"]
    + '"; transcript_biotype "'
    + rna_central_exons["gene_biotype"]
    + '"; exon_id "'
    + rna_central_exons["exon_id"]
    + '"; tag "'
    + rna_central["gene_source"]
    + '";'
)
rna_central_exons["order"] = 3

rna_central_exons["transcript_id"] = rna_central_exons["gene_id"] + ".transcript"
rna_central_exons["exon_coords"] = (
    rna_central_exons[3].astype("str") + "_" + rna_central_exons[4].astype("str") + ","
)
rna_central_gr_transcripts = (
    rna_central_exons.groupby([0, 6, "transcript_id"])
    .agg({"exon_coords": sum})
    .reset_index()
)  # here we concatenate exon coordinates
rna_central_gr_transcripts["transcript_alt_id"] = (
    rna_central_gr_transcripts[0].astype("str")
    + "_"
    + rna_central_gr_transcripts[6]
    + "_"
    + rna_central_gr_transcripts["exon_coords"]
)

gtf_df_exons = gtf_df.loc[gtf_df[2] == "exon"].reset_index(drop=True)
gtf_df_exons["transcript_id"] = (
    gtf_df_exons[8]
    .str.split('transcript_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
gtf_df_exons["exon_coords"] = (
    gtf_df_exons[3].astype("str") + "_" + gtf_df_exons[4].astype("str") + ","
)
gtf_df_gr_transcripts = (
    gtf_df_exons.groupby([0, 6, "transcript_id"])
    .agg({"exon_coords": sum})
    .reset_index()
)  # here we concatenate exon coordinates
gtf_df_gr_transcripts["transcript_alt_id"] = (
    gtf_df_gr_transcripts[0].astype("str")
    + "_"
    + gtf_df_gr_transcripts[6]
    + "_"
    + gtf_df_gr_transcripts["exon_coords"]
)
ensembl_transcripts = list(gtf_df_gr_transcripts["transcript_alt_id"].unique())

preserve_transcripts_list = list(
    rna_central_gr_transcripts.loc[
        ~rna_central_gr_transcripts["transcript_alt_id"].isin(ensembl_transcripts)
    ]["transcript_id"].unique()
)  # when a transcript is present in ensemble and RNA central, prioritize ensemble
rna_central_exons = rna_central_exons.loc[
    rna_central_exons["transcript_id"].isin(preserve_transcripts_list)
].reset_index(drop=True)
rna_central["transcript_id"] = rna_central["gene_id"] + ".transcript"
rna_central = rna_central.loc[
    rna_central["transcript_id"].isin(preserve_transcripts_list)
].reset_index(drop=True)

rna_central_exons = rna_central_exons[list(range(0, 9)) + ["gene_id", "order"]]

rna_central_transcripts = (
    rna_central.loc[rna_central[2] == "transcript"].copy().reset_index(drop=True)
)
rna_central_transcripts[8] = (
    'gene_id "'
    + rna_central_transcripts["gene_id"]
    + '"; transcript_id "'
    + rna_central_transcripts["gene_id"]
    + ".transcript"
    + '"; gene_source "'
    + rna_central_transcripts["gene_source"]
    + '"; gene_biotype "'
    + rna_central_transcripts["gene_biotype"]
    + '"; transcript_source "'
    + rna_central_transcripts["gene_source"]
    + '"; transcript_biotype "'
    + rna_central_transcripts["gene_biotype"]
    + '"; tag "'
    + rna_central_transcripts["gene_source"]
    + '";'
)
rna_central_transcripts["order"] = 2
rna_central_transcripts = rna_central_transcripts[
    list(range(0, 9)) + ["gene_id", "order"]
]

rna_central_genes = (
    rna_central.loc[rna_central[2] == "transcript"].copy().reset_index(drop=True)
)
rna_central_genes[2] = "gene"
rna_central_genes[8] = (
    'gene_id "'
    + rna_central_genes["gene_id"]
    + '"; gene_source "'
    + rna_central_genes["gene_source"]
    + '"; gene_biotype "'
    + rna_central_genes["gene_biotype"]
    + '";'
)
rna_central_genes["order"] = 1
rna_central_genes = rna_central_genes[list(range(0, 9)) + ["gene_id", "order"]]

rna_central_gtf = (
    pd.concat([rna_central_genes, rna_central_transcripts, rna_central_exons])
    .sort_values(["gene_id", "order"])
    .reset_index(drop=True)
    .drop(["gene_id", "order"], 1)
)

enriched_gtf = pd.concat([gtf_df, rna_central_gtf]).reset_index(drop=True)

# TIF-seq data from SGD


def determine_utr(x, strand):
    up_dist = x["utr_start"] - x["start"]
    end_dist = x["end"] - x["utr_end"]
    diff = end_dist - up_dist
    if strand == "+":
        if diff > 0:
            return "5UTR"
        else:
            return "3UTR"
    else:
        if diff > 0:
            return "3UTR"
        else:
            return "5UTR"


tif_seq_gal = pd.read_csv(
    subdirs["annotation_dir"] + "longest_full-ORF_transcripts_gal.gff3",
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=3,
)
tif_seq_ypd = pd.read_csv(
    subdirs["annotation_dir"] + "longest_full-ORF_transcripts_ypd.gff3",
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=3,
)

tif_seq_gal["gene_id"] = (
    tif_seq_gal[8].str.split(";ID=", expand=True)[1].str.split("_", expand=True)[0]
)
tif_seq_ypd["gene_id"] = (
    tif_seq_ypd[8].str.split(";ID=", expand=True)[1].str.split("_", expand=True)[0]
)

tif_seq = pd.merge(
    tif_seq_ypd.rename(columns={3: "start_ypd", 4: "end_ypd"})[
        ["gene_id", 0, 6, "start_ypd", "end_ypd"]
    ],
    tif_seq_gal.rename(columns={3: "start_gal", 4: "end_gal"})[
        ["gene_id", 0, 6, "start_gal", "end_gal"]
    ],
    how="outer",
    on=["gene_id", 0, 6],
)
tif_seq["start"] = tif_seq[["start_ypd", "start_gal"]].min(1).astype("int") - 1
tif_seq["end"] = tif_seq[["end_ypd", "end_gal"]].max(1).astype("int")

tif_seq["score"] = 0
tif_seq = tif_seq[[0, "start", "end", "gene_id", "score", 6]]
tif_seq[0] = tif_seq[0].str.replace("chr", "")

# produce a bed file
a = []
for strand in ["+", "-"]:

    enriched_gtf_bed = (
        enriched_gtf.loc[(enriched_gtf[2] == "gene") & (enriched_gtf[6] == strand)]
        .copy()
        .reset_index(drop=True)
    )
    enriched_gtf_bed = enriched_gtf_bed[[0, 3, 4]]
    enriched_gtf_bed[3] = enriched_gtf_bed[3] - 1
    enriched_gtf_bed.to_csv(
        subdirs["annotation_dir"] + "enriched_gtf.bed",
        sep=str("\t"),
        header=False,
        index=None,
    )
    command = (
        "bedtools sort -i "
        + subdirs["annotation_dir"]
        + "enriched_gtf.bed"
        + " | bedtools merge > "
        + subdirs["annotation_dir"]
        + "enriched_gtf_merged.bed"
    )
    out = subprocess.check_output(command, shell=True)

    tif_seq.loc[tif_seq[6] == strand].to_csv(
        subdirs["annotation_dir"] + "tif_seq.bed",
        sep=str("\t"),
        header=False,
        index=None,
    )
    command = (
        "bedtools subtract -a "
        + subdirs["annotation_dir"]
        + "tif_seq.bed -b "
        + subdirs["annotation_dir"]
        + "enriched_gtf_merged.bed > "
        + subdirs["annotation_dir"]
        + "utrs_"
        + strand
        + ".bed"
    )
    out = subprocess.check_output(command, shell=True)

    utrs = pd.read_csv(
        subdirs["annotation_dir"] + "utrs_" + strand + ".bed",
        delimiter="\t",
        index_col=None,
        header=None,
    )
    utrs = pd.merge(
        utrs.rename(columns={1: "utr_start", 2: "utr_end", 3: "gene_id"})[
            [0, "utr_start", "utr_end", "gene_id"]
        ],
        tif_seq[[0, "start", "end", "gene_id"]],
        how="left",
        on=[0, "gene_id"],
    )
    utrs["UTR_type"] = utrs.apply(lambda x: determine_utr(x, strand), 1)
    utrs = pd.concat(
        [
            utrs.loc[utrs["UTR_type"] == "5UTR"]
            .sort_values(["gene_id", "utr_end"], ascending=[True, True])
            .drop_duplicates(["gene_id"], keep=("last" if strand == "+" else "first")),
            utrs.loc[utrs["UTR_type"] == "3UTR"]
            .sort_values(["gene_id", "utr_start"], ascending=[True, True])
            .drop_duplicates(["gene_id"], keep=("first" if strand == "+" else "last")),
        ]
    ).reset_index(drop=True)
    utrs["gene_id_extended"] = (
        utrs["gene_id"] + "_" + utrs["UTR_type"]
    )  # rename gene as gene_id_UTRtype
    utrs["strand"] = strand
    utrs = utrs[[0, "utr_start", "utr_end", "gene_id", "gene_id_extended", "strand"]]
    a.append(utrs)
utrs = pd.concat(a).reset_index(drop=True)
utrs["utr_start"] = utrs["utr_start"] + 1

gene_biotypes = enriched_gtf.loc[enriched_gtf[2] == "gene"][[8]]
gene_biotypes["gene_biotype"] = (
    gene_biotypes[8]
    .str.split('gene_biotype "', expand=True)[1]
    .str.split('";', expand=True)[0]
)
gene_biotypes["gene_id"] = (
    gene_biotypes[8]
    .str.split('gene_id "', expand=True)[1]
    .str.split('";', expand=True)[0]
)

utrs = pd.merge(
    utrs, gene_biotypes[["gene_id", "gene_biotype"]], how="left", on=["gene_id"]
)

utrs[1] = "tif_seq"
utrs[3] = utrs["utr_start"]
utrs[4] = utrs["utr_end"]
utrs[5] = "."
utrs[6] = utrs["strand"]
utrs[7] = "."

utrs_exons = utrs.copy()
utrs_exons[2] = "exon"
utrs_exons[8] = (
    'gene_id "'
    + utrs_exons["gene_id_extended"]
    + '"; transcript_id "'
    + utrs_exons["gene_id_extended"]
    + ".transcript"
    + '"; exon_number "1"'
    + '; gene_source "tif_seq"; '
    + 'gene_biotype "'
    + utrs_exons["gene_biotype"]
    + '"; transcript_source "tif_seq"; transcript_biotype "'
    + utrs_exons["gene_biotype"]
    + '"; exon_id "'
    + utrs_exons["gene_id_extended"]
    + '.transcript.1"; tag "tif_seq";'
)
utrs_exons["order"] = 3
utrs_exons = utrs_exons[list(range(0, 9)) + ["gene_id_extended", "order"]]

utrs_transcripts = utrs.copy()
utrs_transcripts[2] = "transcript"
utrs_transcripts[8] = (
    'gene_id "'
    + utrs_transcripts["gene_id_extended"]
    + '"; transcript_id "'
    + utrs_transcripts["gene_id_extended"]
    + ".transcript"
    + '"; gene_source "tif_seq"; gene_biotype "'
    + utrs_transcripts["gene_biotype"]
    + '"; transcript_source "tif_seq"; transcript_biotype "'
    + utrs_transcripts["gene_biotype"]
    + '"; tag "tif_seq";'
)
utrs_transcripts["order"] = 2
utrs_transcripts = utrs_transcripts[list(range(0, 9)) + ["gene_id_extended", "order"]]

utrs_genes = utrs.copy()
utrs_genes[2] = "gene"
utrs_genes[8] = (
    'gene_id "'
    + utrs_genes["gene_id_extended"]
    + '"; gene_source "tif_seq"; gene_biotype "'
    + utrs_genes["gene_biotype"]
    + '";'
)
utrs_genes["order"] = 1
utrs_genes = utrs_genes[list(range(0, 9)) + ["gene_id_extended", "order"]]

utrs_gtf = (
    pd.concat([utrs_genes, utrs_transcripts, utrs_exons])
    .sort_values(["gene_id_extended", "order"])
    .reset_index(drop=True)
    .drop(["gene_id_extended", "order"], 1)
)

enriched_gtf = pd.concat([enriched_gtf, utrs_gtf]).reset_index(drop=True)

command = "samtools faidx " + file_paths["genome_file"]
out = subprocess.check_output(command, shell=True)

genome_fai = pd.read_csv(
    file_paths["genome_file"] + ".fai", delimiter="\t", index_col=None, header=None
)
enriched_gtf = pd.merge(genome_fai[[0]], enriched_gtf, how="left", on=0)

enriched_gtf = enriched_gtf.loc[enriched_gtf[2] != "five_prime_utr"].reset_index(
    drop=True
)

enriched_gtf.to_csv(
    file_paths["enriched_annotation_file"],
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

## Obtain bed file with annotated codons in protein-coding transcripts

### v1

In [ ]:
WF_version = "v1"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/modified_annotation/"
command = "mkdir -p " + dir_path

out = subprocess.check_output(command, shell=True)

CDS_and_UTRs_gtf = pd.read_csv(
    dir_path + "CDS_and_UTRs.gtf", delimiter="\t", index_col=None, header=None
)
CDS_and_UTRs_gtf["transcript_id"] = (
    CDS_and_UTRs_gtf[8]
    .str.split('transcript_id "', expand=True)[1]
    .str.split('"', expand=True)[0]
)
CDS_exons = CDS_and_UTRs_gtf.loc[
    CDS_and_UTRs_gtf["transcript_id"].str.endswith("___CDS")
].reset_index(drop=True)
CDS_exons["exon_number"] = (
    CDS_exons[8]
    .str.split('exon_number "', expand=True)[1]
    .str.split('"', expand=True)[0]
    .astype("int")
)

In [ ]:
start_time = time.time()


def get_entries(L, strand, group_df, k, Length, start_time):
    prev_transcript = None
    prev_phase = 2
    start_positions, end_positions, phases, transcript_ids, codons = [], [], [], [], []
    i = 0
    for elem in group_df[["transcript_id", 3, 4]].values:
        if prev_transcript == None or prev_transcript != elem[0]:
            prev_phase = 2
            prev_codon = 0
        cur_start_positions = list(range(elem[1], elem[2]))
        if strand == "-":
            cur_start_positions = cur_start_positions[::-1]
        start_positions = start_positions + cur_start_positions
        end_positions = end_positions + [s + 1 for s in cur_start_positions]
        cur_phases = []
        for s in cur_start_positions:
            phase = (prev_phase + 1) % 3
            cur_phases.append(phase)
            prev_phase = phase
        phases = phases + cur_phases
        cur_codons = []
        for s in cur_phases:
            if s == 0:
                cur_codon = prev_codon + 1
            else:
                cur_codon = prev_codon
            cur_codons.append(cur_codon)
            prev_codon = cur_codon
        codons = codons + cur_codons
        transcript_ids = transcript_ids + [elem[0]] * len(cur_start_positions)
        prev_transcript = elem[0]
        i = i + 1
    entries = np.matrix(
        [start_positions, end_positions, transcript_ids, phases, codons]
    ).transpose()
    L.append(entries)
    if k % 200 == 0 and k != 0:
        print(
            str(k)
            + " out of "
            + str(Length)
            + " done, "
            + str(time.time() - start_time)
        )


with Manager() as manager:
    L = manager.list()
    processes = []
    data = (
        CDS_exons.sort_values([6, "transcript_id", "exon_number", 3, 4])
        .reset_index(drop=True)
        .copy()
    )  # importantly, all elements are sorted in ascending order for exon number
    data[3] = data[3] - 1  # stick to bed format
    Length = len(data["transcript_id"].unique())
    k = 0
    for group_name, group_df in data.groupby([6, "transcript_id"]):
        p = Process(
            target=get_entries, args=(L, group_name[0], group_df, k, Length, start_time)
        )  # Passing the list
        p.start()
        processes.append(p)
        k = k + 1
    for p in processes:
        p.join()
    L = list(L)
res_df = pd.DataFrame(np.concatenate(L, axis=0))
res_df[[0, 1, 3, 4]] = res_df[[0, 1, 3, 4]].astype("int")

In [ ]:
# remove codons near the start and near the end of CDS - those are not informative of translation intensity
if "codon_number" in list(res_df.columns):
    res_df = res_df.drop(["codon_number"], axis=1)
res_df = pd.merge(
    res_df,
    res_df.groupby([2]).agg({4: max}).reset_index().rename(columns={4: "codon_number"}),
    how="left",
    on=[2],
)

exclude_start, exclude_end = 5, 4
selected_codons_df = res_df.loc[
    (res_df[4] >= exclude_start) & (res_df[4] <= (res_df["codon_number"] - exclude_end))
].reset_index(drop=True)

In [ ]:
selected_codons_df["name"] = (
    selected_codons_df[2] + ";" + selected_codons_df[4].astype("str")
)
bed_codon_annotation = pd.merge(
    CDS_exons[[0, "transcript_id", 6]].drop_duplicates(),
    selected_codons_df.rename(columns={0: 1, 1: 2, 2: "transcript_id"})[
        [1, 2, "name", 3, "transcript_id"]
    ],
    how="inner",
    on=["transcript_id"],
)
bed_codon_annotation = bed_codon_annotation[[0, 1, 2, "name", 3, 6]]
# we've stored phase in the 4th column and codon number in the name
# codon number might not be stored in 4th column because bed file convension is often 1-1000 range for this column, so very long proteins may not fit

In [ ]:
# assign weight for overlapping positions (number of overlapping transcripts) - count should then be divided by it

bed_codon_annotation["t"] = 1
bed_codon_annotation = pd.merge(
    bed_codon_annotation.drop(["t"], axis=1),
    bed_codon_annotation.groupby([0, 2, 6]).agg({"t": sum}).reset_index(),
    how="left",
    on=[0, 2, 6],
)
bed_codon_annotation["name"] = (
    bed_codon_annotation["name"] + ";" + bed_codon_annotation["t"].astype("str")
)
bed_codon_annotation = bed_codon_annotation.drop(["t"], axis=1)

In [ ]:
bed_codon_annotation.loc[bed_codon_annotation[6] == "+"].to_csv(
    subdirs["temp_dir"] + "codon_annotation.plus.bed",
    sep=str("\t"),
    header=False,
    index=None,
)
bed_codon_annotation.loc[bed_codon_annotation[6] == "-"].to_csv(
    subdirs["temp_dir"] + "codon_annotation.minus.bed",
    sep=str("\t"),
    header=False,
    index=None,
)

command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + "codon_annotation.plus.bed > "
    + dir_path
    + "codon_annotation.plus.sorted.bed"
)
out = subprocess.check_output(command, shell=True)
command = (
    "bedtools sort -i "
    + subdirs["temp_dir"]
    + "codon_annotation.minus.bed > "
    + dir_path
    + "codon_annotation.minus.sorted.bed"
)
out = subprocess.check_output(command, shell=True)

In [ ]:
dir_path + "codon_annotation.plus.sorted.bed"

# Prepare start samples

In [ ]:
path_to_look = subdirs["input_data_path"] + "20251102*"
os.system(
    """find """
    + path_to_look
    + """ -name '*.fastq.gz' > """
    + subdirs["temp_dir"]
    + """fastq_file_paths.tsv"""
)

In [ ]:
tmp = pd.read_csv(
    subdirs["temp_dir"] + "fastq_file_paths.tsv",
    delimiter="\t",
    index_col=None,
    header=None,
)
tmp.columns = ["path"]
tmp["file_name"] = tmp.apply(lambda x: x["path"].split("/")[-1], 1)
tmp["sample"] = tmp.apply(
    lambda x: "_".join([x["file_name"].split("_")[elem] for elem in [0, 1, 3, 4, 5]]), 1
)  # with lane replicates
tmp["merged_sample"] = tmp.apply(
    lambda x: "_".join([x["file_name"].split("_")[elem] for elem in [0, 1, 3, 4]]), 1
)
tmp["mate"] = tmp.apply(lambda x: "R1" if "_R1_" in x["file_name"] else "R2", 1)
print(len(tmp["merged_sample"].unique()))  # should be 42

fastq_file_paths = tmp.copy()

# Create symbolink link copies for all experimental fastq files

fastq_file_paths["start_file_path"] = (
    subdirs["fastq_dir"]
    + fastq_file_paths["sample"]
    + "."
    + fastq_file_paths["mate"]
    + ".fastq.gz"
)

for index, row in fastq_file_paths.iterrows():
    command = (
        "rm -f "
        + row["start_file_path"]
        + " && ln -f -s "
        + row["path"]
        + " "
        + row["start_file_path"]
    )
    out = subprocess.check_output(command, shell=True)

In [ ]:
# Description of the library prep


https://www.takarabio.com/documents/User%20Manual/SMART/SMART-Seq%20Total%20RNA%20Pico%20Input%20with%20UMIs%20%28ZapR%20Mammalian%29%20User%20Manual.pdf

In [ ]:
WF_version = "v1"

dir_path = subdirs["wf_runs_dir"] + WF_version + "/"
command = "mkdir -p " + dir_path
out = subprocess.check_output(command, shell=True)

start_samples = fastq_file_paths.drop(["path"], axis=1).rename(
    columns={"start_file_path": "fq"}
)

start_samples = pd.merge(
    start_samples.loc[start_samples["mate"] == "R1"][
        ["sample", "merged_sample", "fq"]
    ].rename(columns={"fq": "fq1"}),
    start_samples.loc[start_samples["mate"] == "R2"][
        ["sample", "merged_sample", "fq"]
    ].rename(columns={"fq": "fq2"}),
    how="inner",
    on=["sample", "merged_sample"],
)
start_samples["fq1_3p"] = "AGATCGGAAGAG"  # forcely put Illumina universal adapter
start_samples["fq2_3p"] = "AGATCGGAAGAG"  # forcely put Illumina universal adapter

start_samples["fq2_UMI_len"] = "N" * 8  # first 8nts of read 2 are UMIs
start_samples["fq2_trim5"] = (
    3 + 3
)  # fixed number of nt to trim from 5'end of the R2 reads (template-switching part) AFTER UMI extraction
start_samples["fq1_trim5"] = 0  # nothing to trim

start_samples.to_csv(
    dir_path + "start_samples.tsv",
    sep=str("\t"),
    header=True,
    index=None,
    quoting=csv.QUOTE_NONE,
)

# Prepare .yaml config file and run WF

In [ ]:
# run snakemake under created conda environment snakemake:
# conda activate snakemake

In [ ]:
# install slurm executor
# pip install snakemake-executor-plugin-slurm

In [ ]:
WF_version = "v1"
organism = "yeast"

gtf_chrs = pd.read_csv(
    file_paths["annotation_file"],
    delimiter="\t",
    index_col=None,
    header=None,
    usecols=[0],
    skiprows=5,
)
chromosome_list = list(gtf_chrs[0].unique())

In [ ]:
# load default rule_config, modify it and save
WF_version = "v1"
organism = "yeast"

yaml = ruamel.yaml.YAML()
yaml.preserve_quotes = True
with open(subdirs["wf_dir"] + "config.yaml") as f_read:
    data = yaml.load(f_read)

data["samples_file"] = subdirs["wf_runs_dir"] + WF_version + "/start_samples.tsv"
data["output_dir"] = subdirs["wf_runs_dir"] + WF_version + "/output/"
data["local_log"] = subdirs["wf_runs_dir"] + WF_version + "/output/local_log/"
data["cluster_log"] = subdirs["wf_runs_dir"] + WF_version + "/output/cluster_log/"

data["organism"] = organism
data["genome_file"] = file_paths["genome_file"]
data["gtf_file"] = file_paths["annotation_file"]
data["gtf_CDS_and_UTRs"] = (
    subdirs["wf_runs_dir"] + WF_version + "/modified_annotation/CDS_and_UTRs.gtf"
)

data["chromosomes"] = " ".join(chromosome_list)

for dir_path in [data["output_dir"], data["local_log"], data["cluster_log"]]:
    command = "mkdir -p " + dir_path
    out = subprocess.check_output(command, shell=True)

with open(
    subdirs["wf_runs_dir"] + WF_version + "/modified_config_RNAseq.yaml", "w"
) as f_write:
    yaml.dump(data, f_write)

WF_step = "basic"

command = (
    """snakemake \
--snakefile """
    + subdirs["wf_dir"]
    + """Snakefile-"""
    + WF_step
    + """ \
--scheduler greedy \
--configfile """
    + subdirs["wf_runs_dir"]
    + WF_version
    + "/modified_config_RNAseq.yaml"
    + """ \
--printshellcmds \
--software-deployment-method conda apptainer \
--conda-frontend conda \
--apptainer-args "--bind """
    + subdirs["wf_dir"]
    + """,/scicore/home/zavolan/GROUP/,"""
    + subdirs["input_data_path"]
    + """" \
--executor slurm \
--profile """
    + subdirs["wf_dir"]
    + "profile"
    + """ \
--nolock \
-np"""
)

print(command)

In [ ]:
# TO DO - make coverage smoother by filling in gaps and removing overlaps!

## Define metadata labels and colors uniformly

In [ ]:
def get_metadata_with_labels_and_colors(input_df):
    sel_files = input_df.copy()
    sel_files["fraction"] = sel_files.apply(
        lambda x: x["merged_sample"].split("-")[2], 1
    )
    present_conditions = {
        "D": "minusD",
        "DTT": "DTT",
        "U": "UT",
        "CCCP": "CCCP",
        "H2O2": "H2O2",
        "DMSO": "DMSO",
        "Tun": "Tun",
    }

    def get_condition(x):
        cond_raw = x["merged_sample"].split("-")[5].split("_")[0]
        if cond_raw in present_conditions.keys():
            return present_conditions[cond_raw]
        else:
            cur_cond = ""
            for elem in list(present_conditions.keys()):
                if cond_raw.startswith(elem) and len(elem) > len(cur_cond):
                    cur_cond = elem
            return present_conditions[cur_cond]

    def get_replicate(x):
        unadj = x["merged_sample"].split("_S")[-2]
        if (x["condition"] in ["DMSO", "UT"]) and x["fraction"] == "PD":
            return "R" + unadj.split("-")[-2][-1]
        else:
            return "R" + unadj[-1]

    sel_files["condition"] = sel_files.apply(lambda x: get_condition(x), 1)
    sel_files["replicate"] = sel_files.apply(lambda x: get_replicate(x), 1)
    sel_files[["merged_sample", "fraction", "condition", "replicate"]].drop_duplicates()

    sel_files["sample_label_within_experiment"] = (
        sel_files["fraction"]
        + ";"
        + sel_files["condition"]
        + ";"
        + sel_files["replicate"]
    )

    cat = "condition"
    cat_order = ["UT", "DMSO", "minusD", "DTT", "H2O2", "CCCP", "Tun"]
    cat_palette = [
        "grey",
        "skyblue",
        "orangered",
        "lime",
        "royalblue",
        "yellow",
        "fuchsia",
    ]
    cat_color_dict = {}
    for k, cat_val in enumerate(cat_order):
        cat_color_dict[cat_val] = cat_palette[k]
    sel_files[cat + ";color"] = sel_files[cat].map(cat_color_dict)

    sel_files = sel_files.sort_values(
        ["fraction", "condition", "replicate", "merged_sample", "sample"]
    ).reset_index(drop=True)
    return sel_files

# mRNA Expression

## Analysis of read mapping stats 

In [ ]:
WF_version = "v1"
organism = "yeast"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/"

os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.mapping_stats.txt' > """
    + subdirs["temp_dir"]
    + """mapping_stats.files.txt"""
)
os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.success' > """
    + subdirs["temp_dir"]
    + """mapping_stats.success.files.txt"""
)

mapping_stats_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)
mapping_stats_success_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.success.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)

mapping_stats_files["merged_sample"] = mapping_stats_files.apply(
    lambda x: x[0].split("/")[-1].replace(".mapping_stats.txt", ""), 1
)
mapping_stats_success_files["merged_sample"] = mapping_stats_success_files.apply(
    lambda x: x[0].split("/")[-1].replace(".success", ""), 1
)

mapping_stats_files = mapping_stats_files.loc[
    mapping_stats_files["merged_sample"].isin(
        list(mapping_stats_success_files["merged_sample"].unique())
    )
].reset_index(
    drop=True
)  # look only at files with success flag

start_samples = pd.read_csv(
    dir_path + "start_samples.tsv", delimiter="\t", index_col=None, header=0
)
metadata_df = get_metadata_with_labels_and_colors(start_samples)

sel_files = pd.merge(
    mapping_stats_files,
    metadata_df.drop_duplicates("merged_sample"),
    how="right",
    on=["merged_sample"],
)
samples_list = list(sel_files["merged_sample"])  # list of samples

In [ ]:
sub_sel_files = sel_files.copy()

i = 0
res = []
for index, row in sub_sel_files.iterrows():
    # parse custom format
    tmp = pd.read_csv(row[0], delimiter="\t", index_col=None, header=None)
    df_chunk_source, cur_col_name = [], ""
    res_df = None
    for elem in tmp[0].values:
        if elem.startswith(">"):
            # create a df from the previous chunk
            if len(df_chunk_source) > 0 and cur_col_name != "":
                df_chunk = pd.DataFrame(df_chunk_source, columns=[cur_col_name])
                df_chunk[cur_col_name] = df_chunk[cur_col_name].astype("int")
                df_chunk["t"] = 1
                if res_df is None:
                    res_df = df_chunk[[cur_col_name, "t"]]
                else:
                    res_df = pd.merge(
                        res_df, df_chunk[[cur_col_name, "t"]], how="inner", on="t"
                    )
            cur_col_name = elem[1:]
            df_chunk_source = []
        else:
            df_chunk_source.append(elem)
    if len(df_chunk_source) > 0 and cur_col_name != "":
        df_chunk = pd.DataFrame(df_chunk_source, columns=[cur_col_name])
        df_chunk[cur_col_name] = df_chunk[cur_col_name].astype("int")
        df_chunk["t"] = 1
        if res_df is None:
            res_df = df_chunk[[cur_col_name, "t"]]
        else:
            res_df = pd.merge(
                res_df, df_chunk[[cur_col_name, "t"]], how="inner", on="t"
            )
    res_df["sample"] = row["sample"]
    res.append(res_df.drop(["t"], axis=1))
    if i % 2 == 0 and i != 0:
        print(str(i) + " done")
    i = i + 1
mapping_stats_df = pd.concat(res).reset_index(drop=True)
mapping_stats_df = pd.merge(sel_files, mapping_stats_df, how="left", on="sample")
mapping_stats_df["number of MM reads; genomic mapping"] = (
    mapping_stats_df["total number of mapped reads; genomic mapping"]
    - mapping_stats_df["number of UM reads; genomic mapping"]
)
mapping_stats_df["number of MM reads; UMI dedup"] = (
    mapping_stats_df["total number of mapped reads; UMI dedup"]
    - mapping_stats_df["number of UM reads; UMI dedup"]
)

numb_cols = list(mapping_stats_df.columns)[-6:]
for elem in numb_cols:
    mapping_stats_df[elem + "; mln"] = np.round(mapping_stats_df[elem] / 10**6, 2)

mapping_stats_df["perc_of_UM; genomic mapping"] = (
    mapping_stats_df["number of UM reads; genomic mapping"]
    / mapping_stats_df["total number of mapped reads; genomic mapping"]
    * 100
)
mapping_stats_df["perc_of_UM; UMI dedup"] = (
    mapping_stats_df["number of UM reads; UMI dedup"]
    / mapping_stats_df["total number of mapped reads; UMI dedup"]
    * 100
)

In [ ]:
features = [
    "total number of mapped reads; genomic mapping; mln",
    "number of UM reads; genomic mapping; mln",
    "total number of mapped reads; UMI dedup; mln",
    "number of UM reads; UMI dedup; mln",
]
palette = ["black", "green", "grey", "royalblue"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 8))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "sample_label_within_experiment"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    # ax.set_yticklabels(mapping_stats_df['sample_label_within_experiment'])
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="# reads, mln")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.read_numbers.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.read_numbers.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
mapping_stats_df['number of UM reads; UMI dedup; mln'].quantile(0.5)

In [ ]:
mapping_stats_df[['sample','number of UM reads; UMI dedup; mln']].sort_values('number of UM reads; UMI dedup; mln')

In [ ]:
features = ["perc_of_UM; genomic mapping", "perc_of_UM; UMI dedup"]
palette = ["green", "royalblue"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 8))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "sample_label_within_experiment"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="% of mapped reads")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.perc_of_UM.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.perc_of_UM.pdf",
    bbox_inches="tight",
    dpi=300,
)

## merged_sample - level counts

In [ ]:
WF_version = "v1"
organism = "yeast"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/"

os.system(
    """find """
    + dir_path
    + "output/gene_expression_quantification/FeatureCounts_genomic_DefaultGTF/"
    + """ -name '*.txt' > """
    + subdirs["temp_dir"]
    + """FeatureCounts_genomic_DefaultGTF.files.txt"""
)
FeatureCounts_files = pd.read_csv(
    subdirs["temp_dir"] + "FeatureCounts_genomic_DefaultGTF.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)
FeatureCounts_files["merged_sample"] = FeatureCounts_files.apply(
    lambda x: x[0].split("/")[-1].replace(".txt", ""), 1
)

start_samples = pd.read_csv(
    dir_path + "start_samples.tsv", delimiter="\t", index_col=None, header=0
)
metadata_df = get_metadata_with_labels_and_colors(start_samples)

sel_files = pd.merge(
    FeatureCounts_files,
    metadata_df.drop_duplicates("merged_sample"),
    how="right",
    on=["merged_sample"],
)
samples_list = list(sel_files["merged_sample"])  # list of samples

In [ ]:
len(sel_files), len(list(start_samples["merged_sample"].unique()))

In [ ]:
gtf = pd.read_csv(
    file_paths["annotation_file"],
    delimiter="\t",
    index_col=None,
    header=None,
    skiprows=5,
)
gtf["gene_id"] = (
    gtf[8].str.split('gene_id "', expand=True)[1].str.split('"', expand=True)[0]
)
gtf["gene_name"] = (
    gtf[8].str.split('gene_name "', expand=True)[1].str.split('"', expand=True)[0]
)
gtf["gene_type"] = (
    gtf[8].str.split('gene_biotype "', expand=True)[1].str.split('"', expand=True)[0]
)
genes = (
    gtf.loc[gtf[2] == "gene"][["gene_id", "gene_name", "gene_type"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
# load raw counts from FeatureCount outputs

i = 0
a = []
for index, row in sel_files.iterrows():
    if i == 0:
        row_names = pd.read_csv(
            row[0],
            delimiter="\t",
            index_col=None,
            header=None,
            skiprows=2,
            usecols=[0, 1, 2, 3, 4, 5],
        )
    tmp = pd.read_csv(
        row[0], delimiter="\t", index_col=None, header=None, skiprows=2, usecols=[6]
    )

    a.append(tmp[6])  # take advantage of the fact that the order is the same
    if i % 100 == 0 and i != 0:
        print(str(i) + " done")
    i = i + 1

a = np.matrix(a).transpose()
res_FC = pd.DataFrame(a, columns=list(sel_files["merged_sample"]))

row_names.columns = ["gene_id", "chr", "start", "end", "strand", "gene_length"]

res_FC = pd.concat([row_names, res_FC], axis=1)
res_FC = pd.merge(genes, res_FC, how="right", on="gene_id")

res_FC[samples_list] = np.round(res_FC[samples_list] + 1, 0).astype(
    "int"
)  # pseudocount

In [ ]:
# normalize by library size as Deseq2 does
# but anchor to TOTAL samples!
# total_samples_list = [elem for elem in samples_list if "-Total-" in elem]
total_samples_list = samples_list # anchoring to median of all samples

res_FC["median"] = res_FC[total_samples_list].median(1)
res_FC["q10"] = res_FC[total_samples_list].quantile(q=0.1, axis=1)
res_FC["mean"] = res_FC[total_samples_list].mean(1)

tmp = res_FC.loc[res_FC["median"] > res_FC["median"].quantile(0.7)].reset_index(
    drop=True
)
tmp[samples_list] = tmp[samples_list].div(tmp["median"], axis=0)
sfs = pd.DataFrame(tmp[samples_list].median())
sfs["merged_sample"] = sfs.index
sfs = sfs.rename(columns={0: "sf"})
sfs["log2_sf"] = np.log2(sfs["sf"])

Rsum = pd.DataFrame((res_FC[samples_list] - 1).sum(), columns=["read_sum"])
Rsum["merged_sample"] = Rsum.index
sfs = pd.merge(sfs, Rsum, how="left", on="merged_sample")
sfs.index = sfs["merged_sample"]

norm_log_counts = res_FC.copy()
norm_log_counts[samples_list] = np.log2(
    norm_log_counts[samples_list].div(sfs["sf"], axis=1)
)

sfs = sfs.reset_index(drop=True)

In [ ]:
sfs["read_sum_mln"] = np.round(sfs["read_sum"] / 10**6, 2)

alpha_param, s_param = 0.7, 40
sns.set(font_scale=1.2)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(4, 4))

x_feature, y_feature = "read_sum_mln", "sf"

hue = "fraction"
hue_order = ["PD", "Total"]
palette = ["orange", "teal"]

data = pd.merge(sfs, sel_files, how="right", on="merged_sample")

ax = sns.scatterplot(
    ax=axes,
    data=data,
    x=x_feature,
    y=y_feature,
    s=s_param,
    alpha=alpha_param,
    edgecolor="black",
    linewidth=0.5,
    hue=hue,
    hue_order=hue_order,
    palette=palette,
)

ax.set(xlabel="# reads, mln")
ax.set(ylabel="size factor,\nDeseq2 normalization")
ax.tick_params(left=True, bottom=True)

ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)

texts = []
for index, row in data.iterrows():
    texts.append(
        ax.annotate(
            xy=(row[x_feature], row[y_feature]),
            text=row["replicate"],
            size=10,
            ha="left",
        )
    )
%time adjust_text(texts,arrowprops=dict(arrowstyle='-', color='black'),ax=ax,force_points=(5,5))
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "gene_expression/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/library_size_vs_SF.png",
    bbox_inches="tight",
    dpi=600,
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/library_size_vs_SF.pdf",
    bbox_inches="tight",
    dpi=600,
)

In [ ]:
# check without correction

norm_log_counts["m"] = norm_log_counts[samples_list].mean(axis=1)
PCA_UMAP_subset = norm_log_counts.loc[
    norm_log_counts["m"] > norm_log_counts["m"].quantile(0.5)
].reset_index(drop=True)

bw_adjust = 1.75
alpha_param, s_param = 0.7, 40
sns.set(font_scale=1.2)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(5, 5))

# PCA with different coloring
x_feature, y_feature = "PC1", "PC2"

hue = "condition"
hue_order = ["UT", "DMSO", "H2O2", "DTT", "Tun", "minusD", "CCCP"]
palette_df = sel_files[[hue, hue + ";color"]].drop_duplicates()
palette = [
    palette_df.loc[palette_df[hue] == elem][hue + ";color"].iloc[0]
    for elem in hue_order
]

x = (PCA_UMAP_subset[samples_list].values).transpose()
x = StandardScaler().fit_transform(x)

pca = PCA(n_components=4)
principalComponents = pca.fit_transform(x)
principalDf = pd.DataFrame(
    data=principalComponents, columns=["PC1", "PC2", "PC3", "PC4"]
)
principalDf["merged_sample"] = samples_list
principalDf = pd.merge(principalDf, sel_files, how="right", on="merged_sample")

ax = sns.scatterplot(
    data=principalDf,
    x=x_feature,
    y=y_feature,
    s=s_param,
    alpha=alpha_param,
    edgecolor="black",
    linewidth=0.5,
    style="fraction",
    hue=hue,
    hue_order=hue_order,
    palette=palette,
)

ax.set(
    xlabel="PC1, "
    + str(int(np.round(pca.explained_variance_ratio_[0] * 100, 0)))
    + "% variance"
)
ax.set(
    ylabel="PC2, "
    + str(int(np.round(pca.explained_variance_ratio_[1] * 100, 0)))
    + "% variance"
)

ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)

texts = []
for index, row in principalDf.iterrows():
    texts.append(
        ax.annotate(
            xy=(row[x_feature], row[y_feature]),
            text=row["replicate"],
            size=10,
            ha="left",
        )
    )
%time adjust_text(texts,arrowprops=dict(arrowstyle='-', color='black'),ax=ax,force_points=(5,5))
ax.tick_params(left=True, bottom=True)

out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "gene_expression/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/PCA_gene_expression.PC1_PC2.png",
    bbox_inches="tight",
    dpi=600,
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/PCA_gene_expression.PC1_PC2.pdf",
    bbox_inches="tight",
    dpi=600,
)

In [ ]:
# run differential analysis
# conda activate deseq

raw_matrix = res_FC.copy()  # prepare input for Deseq2
raw_matrix[samples_list] = (
    raw_matrix[samples_list] - 1
)  # we don't need pseudocount for Deseq2
raw_matrix[samples_list] = np.round(raw_matrix[samples_list]).astype("int")

list_of_output_tables = []
command = ""

exclude_replicates = []

CtrlNonCtrl_conditions = [["DMSO","CCCP"], ['UT',"DMSO"], ['UT',"DTT"], ['UT',"H2O2"], ['DMSO',"Tun"], ['UT',"minusD"]]

for k1, fraction in enumerate(["Total", "PD"]):
    for k2, CtrlNonCtrl_condition in enumerate(CtrlNonCtrl_conditions):
        CtrlCond,NonCtrlCond = CtrlNonCtrl_condition
        sub_sel_metadata = (
            sel_files.loc[
                (~sel_files["replicate"].isin(exclude_replicates))
                & (sel_files["fraction"] == fraction)
                & (sel_files["condition"].isin([CtrlCond, NonCtrlCond]))
            ][["merged_sample", "condition", "replicate"]]
            .reset_index(drop=True)
            .copy()
        )
        if (
            len(sub_sel_metadata) >= 2
            and len(sub_sel_metadata["condition"].unique()) == 2
        ):
            sub_sel_metadata["EXP_CTL"] = sub_sel_metadata.apply(
                lambda x: "EXP" if x["condition"] == NonCtrlCond else "CTL", 1
            )
            index_cols = ["gene_id"]
            sub_samples_list = list(sub_sel_metadata["merged_sample"])
            cur_res = raw_matrix[index_cols + sub_samples_list].reset_index(drop=True)
            cur_res[sub_samples_list] = (
                cur_res[sub_samples_list].fillna(0).astype("int")
            )
            cur_res = (
                cur_res[["gene_id"] + sub_samples_list]
                .drop_duplicates("gene_id")
                .reset_index(drop=True)
            )  # to double-check that there are no duplicate entries

            prefix_path = (
                subdirs["tables_dir"]
                + "gene_expression/"
                + fraction
                + "/"
                + NonCtrlCond + "_vs_" + CtrlCond
                + "/"
            )

            out = subprocess.check_output(
                "mkdir -p " + prefix_path,
                shell=True,
            )

            counts_file_path = prefix_path + "counts.tsv"
            metadata_file_path = prefix_path + "metadata.tsv"
            cur_res.to_csv(
                counts_file_path,
                sep=str("\t"),
                header=True,
                index=None,
                quoting=csv.QUOTE_NONE,
            )

            # default Deseq2 model, no confounders
            metadata_cols = ["EXP_CTL"]
            for col in ["confounder_1", "confounder_2", "confounder_3"]:
                sub_sel_metadata[col] = 1
                metadata_cols.append(col)
            sub_sel_metadata[["merged_sample"] + metadata_cols].to_csv(
                metadata_file_path,
                sep=str("\t"),
                header=True,
                index=None,
                quoting=csv.QUOTE_NONE,
            )

            outdir_path = prefix_path
            out = subprocess.check_output("mkdir -p " + outdir_path, shell=True)
            command = (
                command
                + """Rscript """
                + subdirs["wf_dir"]
                + "scripts/run_deseq2.r"
                + " "
                + counts_file_path
                + " "
                + metadata_file_path
                + " EXP "
                + outdir_path
                + " fake_gene no_confounders; "
            )
            list_of_output_tables.append(
                [fraction, NonCtrlCond, CtrlCond, "no_confounders", outdir_path + "DE_table.tsv"]
            )
command

In [ ]:
deseq_results_links = pd.DataFrame(
    list_of_output_tables, columns=["fraction", "condition",'CTRL', "confounders", "path"]
)

In [ ]:
norm_log_counts.loc[norm_log_counts['gene_name']=='BSC1'][list(sel_files.loc[(sel_files['condition'].isin(['UT','H2O2']))&(sel_files['fraction']=='Total')]['merged_sample'])]

In [ ]:
genes.loc[genes['gene_name'].astype('str').str.contains('|'.join(['DCP2','RPS9','RPL19B','RPS6A','RPL19B','BSC1','RPL39','ACT1']))]

In [ ]:
genes_of_interest_list = [
    "ACT1",
    "PGK1",
    "BSC1",
    "DCP2",
    "TPI1",
    "RLM1",
    "ATP11",
    "MET10",
    # "ILM1",
    # "MRPL38",
    # "AIM2",
    # "PTP3",
    # "NSL1",
    "RPS6A",
    # "RPS9A",
    # "RPS9B",
    "RPL19B",
]

confounders = "no_confounders"

x_feature, y_feature, hue = "log2_baseMean", "log2FoldChange", "DE_labelled"
alpha_param, s_param = 0.3, 5
cat1, cat2 = "fraction", "condition"

deseq_results = []  # list to store the Deseq output tables

deseq_results_links_cur = deseq_results_links.loc[
    deseq_results_links["confounders"] == confounders
].reset_index(drop=True)

sns.set(font_scale=0.8)
sns.set_style("white")
fig, axes = plt.subplots(
    len(deseq_results_links_cur[cat1].unique()),
    len(deseq_results_links_cur[cat2].unique()),
    sharey=False,
    sharex=False,
    figsize=(
        4 * len(deseq_results_links_cur[cat2].unique()),
        4 * len(deseq_results_links_cur[cat1].unique()),
    ),
)

for i, cat1_val in enumerate(["Total", "PD"]):
    for j, cat2_val in enumerate(["DMSO", "H2O2", "DTT", "Tun", "minusD", "CCCP"]):
        row = deseq_results_links_cur.loc[
            (deseq_results_links_cur[cat1] == cat1_val)
            & (deseq_results_links_cur[cat2] == cat2_val)
        ]
        if len(row) > 0:
            row = row.iloc[0]
            CtrlCond = row['CTRL']
            if os.path.isfile(row["path"]):
                data = pd.read_csv(
                    row["path"], delimiter="\t", index_col=None, header=0
                )
                data[["pvalue", "padj"]] = data[["pvalue", "padj"]].fillna(1)
                data["log2FoldChange"] = data["log2FoldChange"].fillna(0)
                data["log2_baseMean"] = np.log2(data["baseMean"] + 1)
                data["DE"] = (data["padj"] < 0.05).astype("int") * (
                    (data["log2FoldChange"] > 0).astype("int") * 2 - 1
                )
                data = data.rename(columns={"gene_name": "gene_id"})

                data = pd.merge(genes, data, how="right", on="gene_id")

                # significance score
                min_positive_pval = data.loc[data["pvalue"] > 0]["pvalue"].min()
                data["DE_score"] = (
                    (-1)
                    * np.log10(data["pvalue"] + min_positive_pval)
                    * ((data["log2FoldChange"] > 0).astype("int") * 2 - 1)
                )

                data["t"] = 1
                # add abundance of transcripts to the category labels
                gr = data.groupby(["DE"]).agg({"t": sum}).reset_index()
                short_dict = {-1: "down", 0: "NS", 1: "up"}
                relabel_dict = {}
                for index_gr, row_gr in gr.iterrows():
                    relabel_dict[row_gr["DE"]] = (
                        short_dict[row_gr["DE"]]
                        + ", "
                        + str(np.round(row_gr["t"] / 1000, 2))
                        + " K genes"
                    )
                data[hue] = data["DE"].map(relabel_dict).astype("category")

                data["confounding_factor"] = confounders
                data[cat1] = cat1_val
                data[cat2] = cat2_val
                data['CTRL'] = CtrlCond
                deseq_results.append(data.copy())

                gr[hue] = gr["DE"].map(relabel_dict).astype("category")
                gr = gr.sort_values("DE", ascending=False).reset_index(drop=True)

                hue_order = list(gr[hue])
                palette = ["red", "grey", "blue"]

                # data1 = data.loc[data['DE']!=0]
                # data2 = data.loc[data['DE']==0].sample(min(len(data1)*3,len(data)/2))
                # data_sampled = pd.concat([data1,data2]).reset_index(drop=True)
                data_sampled = data.copy()

                ### MA-plot
                ax = sns.scatterplot(
                    ax=(
                        axes[i][j]
                        if (len(deseq_results_links_cur[cat1].unique()) > 1)
                        else axes[0]
                    ),
                    data=data_sampled,
                    x=x_feature,
                    y=y_feature,
                    hue=hue,
                    hue_order=hue_order,
                    palette=palette,
                    alpha=alpha_param,
                    s=s_param,
                    legend=True,
                )

                # plot selected genes
                interesting_subset = data.loc[
                    (data["gene_name"].isin(genes_of_interest_list))
                ].reset_index(drop=True)
                ax = sns.scatterplot(
                    ax=(
                        axes[i][j]
                        if (len(deseq_results_links_cur[cat1].unique()) > 1)
                        else axes[0]
                    ),
                    data=interesting_subset,
                    x=x_feature,
                    y=y_feature,
                    hue=hue,
                    hue_order=hue_order,
                    palette=palette,
                    legend=False,
                    alpha=1,
                    s=20,
                    edgecolor="black",
                    linewidth=2,
                )
                texts = []
                for index, row in interesting_subset.iterrows():
                    texts.append(
                        ax.annotate(
                            xy=(row[x_feature], row[y_feature]),
                            text=row["gene_name"],
                            size=8,
                            ha="left",
                        )
                    )
                %time adjust_text(texts,arrowprops=dict(arrowstyle='-', color='black'),ax=ax,force_text=(13,10))

                xlims = (ax.get_xlim()[0], ax.get_xlim()[1])
                ax.hlines(
                    y=0,
                    xmin=xlims[0],
                    xmax=xlims[1],
                    linestyle="--",
                    color="grey",
                    lw=1.5,
                )

            else:
                ax = sns.scatterplot(
                    ax=axes[i][j],
                    data=pd.DataFrame([[0, 0]], columns=[x_feature, y_feature]),
                    x=x_feature,
                    y=y_feature,
                    color="white",
                    alpha=0,
                    s=s_param,
                )
                ax.annotate(
                    xy=(0, 0),
                    text="DE test is not allowed",
                    size=10,
                    ha="left",
                    va="center",
                )
        else:
            CtrlCond = 'NA'
            ax = sns.scatterplot(
                ax=axes[i][j],
                data=pd.DataFrame([[0, 0]], columns=[x_feature, y_feature]),
                x=x_feature,
                y=y_feature,
                color="white",
                alpha=0,
                s=s_param,
            )
            ax.annotate(
                xy=(0, 0),
                text="experiment not performed",
                size=10,
                ha="left",
                va="center",
            )
        if i == 0:
            ax.set(title=cat2_val+'-vs-'+CtrlCond)
        ax.tick_params(left=True, bottom=True)

        if j == 0:
            ax.set(ylabel=r"$log_2$FC, "+cat1_val)
        else:
            ax.set(ylabel="")
        ax.legend(
            bbox_to_anchor=(0.0, 0.01),
            loc=3,
            borderaxespad=0.0,
            title="DE test:",
            ncols=1,
            markerscale=2,
        )
        ax.set(xlim=(-1, 20), ylim=(-5, 5), xlabel="$log_2$ av. gene expr.")

out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "gene_expression/",
    shell=True,
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/DE_analysis.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/DE_analysis.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
All_deseq_df = pd.concat(deseq_results).reset_index(drop=True)

index_cols = ["gene_id", "gene_name", "gene_type", "confounding_factor", "condition",'CTRL']
feature_cols = ["baseMean","log2FoldChange", "padj",'DE_score']

tmp = All_deseq_df.loc[All_deseq_df["fraction"] == "Total"][index_cols + feature_cols]
rename_dict = {}
for elem in feature_cols:
    rename_dict[elem] = elem + "_Total"
Total_deseq_df = tmp.rename(columns=rename_dict).reset_index(drop=True)

tmp = All_deseq_df.loc[All_deseq_df["fraction"] == "PD"][index_cols + feature_cols]
rename_dict = {}
for elem in feature_cols:
    rename_dict[elem] = elem + "_PD"
PD_deseq_df = tmp.rename(columns=rename_dict).reset_index(drop=True)

correspondence_df = pd.merge(
    Total_deseq_df, PD_deseq_df, how="inner", on=index_cols
).dropna()

In [ ]:
correspondence_df.loc[correspondence_df['gene_name'].isin(['RPS6A','RPL19B'])]

### Analyzing general trends using RnaFracMod

In [ ]:
# conda activate rnafracmod
# submit as sbatch jobs

raw_matrix = res_FC.copy()  # prepare input for RnaFracMod
raw_matrix[samples_list] = (
    raw_matrix[samples_list] - 1
)  # we don't need pseudocount for RnaFracMod
raw_matrix[samples_list] = np.round(raw_matrix[samples_list]).astype("int")

list_of_output_tables = []
command = ""

exclude_replicates = []

CtrlNonCtrl_conditions = [["DMSO","CCCP"], ['UT',"DMSO"], ['UT',"DTT"], ['UT',"H2O2"], ['DMSO',"Tun"], ['UT',"minusD"]]
fractionLabel = 'PD'

for k2, CtrlNonCtrl_condition in enumerate(CtrlNonCtrl_conditions):
    CtrlCond,NonCtrlCond = CtrlNonCtrl_condition
    sub_sel_metadata = (
        sel_files.loc[
            (~sel_files["replicate"].isin(exclude_replicates))
            & (sel_files["condition"].isin([CtrlCond, NonCtrlCond]))
        ][["merged_sample", "fraction", "condition", "replicate"]]
        .reset_index(drop=True)
        .copy()
    )
    if (
        len(sub_sel_metadata) >= 4
        and len(sub_sel_metadata["condition"].unique()) == 2
    ):
        sub_sel_metadata["EXP_CTL"] = sub_sel_metadata.apply(
            lambda x: "EXP" if (x["condition"] == NonCtrlCond) else "CTL", 1
        )
        
        index_cols = ["gene_id"]
        sub_samples_list = list(sub_sel_metadata["merged_sample"])
        cur_res = raw_matrix[index_cols + sub_samples_list].reset_index(drop=True)
        cur_res[sub_samples_list] = (
            cur_res[sub_samples_list].fillna(0).astype("int")
        )
        
        cur_res = (
            cur_res[["gene_id"] + sub_samples_list]
            .drop_duplicates("gene_id")
            .reset_index(drop=True)
        )  # to double-check that there are no duplicate entries

        prefix_path = (
            subdirs["tables_dir"]
            + "gene_expression/"
            + "RnaFracMod/"
            + NonCtrlCond + "_vs_" + CtrlCond
            + "/"
        )

        out = subprocess.check_output(
            "mkdir -p " + prefix_path,
            shell=True,
        )

        counts_file_path = prefix_path + "counts_RnaFracMod.tsv"
        metadata_file_path = prefix_path + "metadata_RnaFracMod.tsv"
        cur_res.to_csv(
            counts_file_path,
            sep=str("\t"),
            header=True,
            index=None,
            quoting=csv.QUOTE_NONE,
        )

        metadata_cols = ["EXP_CTL",'fraction'] 

        sub_sel_metadata = sub_sel_metadata[["merged_sample"] + metadata_cols].rename(columns={'merged_sample':'sample'})
        sub_sel_metadata['fraction'] = sub_sel_metadata['fraction'].map({'PD':'F','Total':'T'})
        sub_sel_metadata.to_csv(
            metadata_file_path,
            sep=str("\t"),
            header=True,
            index=None,
            quoting=csv.QUOTE_NONE,
        )

        outdir_path = prefix_path
        out = subprocess.check_output("mkdir -p " + outdir_path, shell=True)
        command = (
            command
            + "rnafracmod"
            + " --count_matrix " + counts_file_path
            + " --metadata " + metadata_file_path
            + " --outdir " + outdir_path
            + "; "
        )
        list_of_output_tables.append(
            [NonCtrlCond, CtrlCond, outdir_path + "RnaFracMod_FittedParameters.tsv"]
        )
command

In [ ]:
from scipy.stats import entropy
from scipy.optimize import minimize

def quantile_normalize_to_reference(target, reference):
    """
    Normalizes 'target' so it matches the distribution of 'reference'.
    """
    # 1. Sort the reference to get the desired distribution curve
    sorted_reference = np.sort(reference)
    
    # 2. Get the rank of the target values
    # argsort twice gives the rank (0, 1, ... N-1) of each element
    rank_indices = np.argsort(np.argsort(target))
    
    # 3. Calculate percentiles (0.0 to 1.0) for the target ranks
    # Use (N - 1) to ensure the range covers 0.0 to 1.0 inclusive
    target_percentiles = rank_indices / (len(target) - 1)
    
    # 4. Calculate percentiles for the reference data
    reference_percentiles = np.linspace(0, 1, len(reference))
    
    # 5. Interpolate
    # Find what value in the sorted reference corresponds to the target's percentile
    normalized_target = np.interp(target_percentiles, reference_percentiles, sorted_reference)
    
    return normalized_target

def hybrid_poisson_sampler(log_lambdas, cutoff=5):
    """
    Samples from a vector of log2 logarithms of lambdas using Poisson for small values
    and a Normal approximation for large values.
    """
    log_lambdas = np.asanyarray(log_lambdas)
    samples = np.zeros_like(log_lambdas)
    
    # Create masks for small and large lambdas
    is_large = log_lambdas > cutoff
    is_small = ~is_large
    
    # 1. Poisson sampling for small lambdas
    if np.any(is_small):
        samples[is_small] = np.random.poisson(2**log_lambdas[is_small])
        
    # 2. Normal approximation for large lambdas: N(mu=lam, sigma=sqrt(lam))
    if np.any(is_large):
        mu = 2**log_lambdas[is_large]
        sigma = np.sqrt(mu)
        # Use round to keep results as integers (Poisson properties)
        # and clip to 0 to avoid rare negative values from Normal tails
        norm_samples = np.random.normal(loc=mu, scale=sigma)
        samples[is_large] = np.maximum(0, np.round(norm_samples))
        
    return samples

def simulate_log2_nCPM_from_one_condition(TotExpr_df=2,log_TotExpr_loc=5,log_TotExpr_scale=3,
                       SF_s = 1, SF_loc = 0, SF_scale = 1,
                       N_r = 3, N_g = 7000, log2_f_alpha=0,log2_f_beta=0,
                    debug=True):
    # N_r is a number of replicate samples per condition
    # N_g is a number of genes that are truly expressed
    # f_alpha and f_beta define the parameters of Beta distribution specifying the proportion of each 
    # molecule in a sub-fraction
    
    log2_r_1 = stats.t.rvs(df = TotExpr_df,loc=log_TotExpr_loc,scale=log_TotExpr_scale, size=N_g) # distribution of molecule counts per gene
    r_1 = np.round(2**log2_r_1,0)+1 # need at least one molecule per gene, make discrete

    f_alpha,f_beta = 1+2**log2_f_alpha,1+2**log2_f_beta # during optimization we don't use constraints, but we want values below 1 here
    p_1 = stats.beta.rvs(a=f_alpha, b=f_beta, size=N_g) # proportion of molecules in a sub-fraction, condition 1, aka recruitment efficiency (RE)!
    min_positive_p = np.min(p_1[p_1>0])
    p_1 = p_1+min_positive_p # like pseudocount
    
    f_1 = np.round(p_1*r_1,0) # True number of molecules in a fraction, condition 1
    log2_f_1 = np.log2(p_1)+log2_r_1
    
    # here, proportion in a sub-fraction, and total counts are assumed to be independent
    # so, we don't expect e.g. that higher-expressed genes would have higher RE
    
    # now, we do Poisson sampling for defined number of bioreplicates
    expr_data,cols = [],[]
    total_c1_samples,frac_c1_samples = [],[]
    for repl in range(N_r):
        # true log2 size factor is a random variable following Normal Distrib
        sf_1,f_sf_1 = stats.lognorm.rvs(s = SF_s, loc=SF_loc, scale=SF_scale, size=2) # _1 specifies condition 1
        log_sf_1,log_f_sf_1 = np.log(sf_1),np.log(f_sf_1)
        if debug:
            pass
            # print('log size factors: '+str([log_sf_1,log_f_sf_1]))
            
        log_lam_1,log_f_lam_1 = log2_r_1+log_sf_1,log2_f_1+log_f_sf_1 # lambda parameters for count sampling
        if debug:
            pass
            # print('log lambda vals Total: '+str([min(log_lam_1),max(log_lam_1)]))
            # print('log lambda vals SubFrac: '+str([min(log_f_lam_1),max(log_f_lam_1)]))
        
        expr_data.append(hybrid_poisson_sampler(log_lam_1))
        expr_data.append(hybrid_poisson_sampler(log_f_lam_1))
    
        cols = cols + [str(repl)+'_c1_Total',str(repl)+'_c1_Frac']
        total_c1_samples.append(str(repl)+'_c1_Total')
        frac_c1_samples.append(str(repl)+'_c1_Frac')
        
    expr_df = pd.DataFrame(expr_data).transpose()
    expr_df.columns = cols
    
    # do normalization like Deseq2
    expr_df['min_val'] = expr_df[cols].min(1)
    tmp_df = expr_df.loc[expr_df['min_val']>0].reset_index(drop=True)
    tmp_df['geom_mean'] = (tmp_df[cols].prod(1))**(1/len(cols))
    tmp_df[cols] = tmp_df[cols].div(tmp_df['geom_mean'],axis=0)
    sfs = tmp_df[cols].median(0)
    
    norm_expr_df = np.log2((expr_df[cols]+1).div(sfs,axis=1)) # log2, pseudocount=1
    norm_expr_df['log2_nCPM_Total'] = norm_expr_df[total_c1_samples].mean(1)
    norm_expr_df['log2_nCPM_SubFraction'] = norm_expr_df[frac_c1_samples].mean(1)
    
    return norm_expr_df
    
def kl_divergence_hist(samples_p, samples_q, bins=50,show_plot=False,Plot_SuperTitle=''):
    # the higher the number of bins -> the higher will be KL divergence value
    # Determine common range for both histograms
    # samples_p represents reference, while samples_q represents query
    
    combined = np.concatenate([samples_p, samples_q])
    bin_edges = np.linspace(np.min(combined), np.max(combined), bins + 1)
    
    # Calculate histograms (densities)
    p_hist, _ = np.histogram(samples_p, bins=bin_edges, density=True)
    q_hist, _ = np.histogram(samples_q, bins=bin_edges, density=True)
    
    # Avoid zeros
    p_hist += 1e-10
    q_hist += 1e-10
    
    # Normalize to ensure sum is 1
    p_hist /= p_hist.sum()
    q_hist /= q_hist.sum()
    kl_div = entropy(p_hist, q_hist)
    
    if show_plot:
        sns.set(font_scale=0.7)
        sns.set_style("white")
        fig, axes = plt.subplots(1, 1, sharey=True, sharex=True, figsize=(2.5, 1.2))
        ax = sns.histplot(samples_p,bins=bin_edges,color='green',alpha=0.5,stat='density',
                          label='obs')
        ax = sns.histplot(samples_q,bins=bin_edges,color='orange',alpha=0.5,stat='density',
                          label='exp')    
        ax.legend(
            bbox_to_anchor=(1.05, 1.00),
            loc="upper left",
            borderaxespad=0.0,
            title="",
            ncols=1,
            markerscale=5,
        )
        ax.set(xlabel='$log_2$ norm. gene expression\n(observed)',
               ylabel='density',
              title = Plot_SuperTitle+'\n'+'KLdiv='+str(np.round(kl_div,3)))
    return kl_div


# first step is to fit distribution of norm. expr values in Total
def compute_KL_div_with_simulation_for_Total(params=[2,5,3],show_plot=False,Nsimul=10,return_list_of_vals=False):
    TotExpr_df,log_TotExpr_loc,log_TotExpr_scale = params
    kl_divergence_vals = []
    for i in range(Nsimul):
        simul_norm_expr_df = simulate_log2_nCPM_from_one_condition(
            TotExpr_df=TotExpr_df,log_TotExpr_loc=log_TotExpr_loc,log_TotExpr_scale=log_TotExpr_scale,
            SF_s = SF_s, SF_loc = SF_loc, SF_scale = SF_scale,
            N_r = 3, N_g = len(obs_list), log2_f_alpha=0,log2_f_beta=0)
        
        expected_Total = list(simul_norm_expr_df['log2_nCPM_Total'])
        kl_divergence_val = kl_divergence_hist(obs_list,expected_Total,show_plot=show_plot,Plot_SuperTitle = 'Total')
        kl_divergence_vals.append(kl_divergence_val)
    if return_list_of_vals:
        return kl_divergence_vals
    else:
        return np.mean(kl_divergence_vals)

def compute_KL_div_with_simulation_for_SubFraction(params=[0,0],show_plot=False,Nsimul=10):
    log2_f_alpha,log2_f_beta = params
    kl_divergence_vals = []
    for i in range(Nsimul):
        simul_norm_expr_df = simulate_log2_nCPM_from_one_condition(
            TotExpr_df=TotExpr_df,log_TotExpr_loc=log_TotExpr_loc,log_TotExpr_scale=log_TotExpr_scale,
            SF_s = SF_s, SF_loc = SF_loc, SF_scale = SF_scale,
            N_r = 3, N_g = len(obs_list),
            log2_f_alpha=log2_f_alpha,log2_f_beta=log2_f_beta)
        
        expected_SubFraction = list(simul_norm_expr_df['log2_nCPM_SubFraction'])
        kl_divergence_val = kl_divergence_hist(obs_list,expected_SubFraction,show_plot=show_plot,Plot_SuperTitle = 'Sub-fraction')
        kl_divergence_vals.append(kl_divergence_val)
    return np.mean(kl_divergence_vals)

# multi-start local optimization, to increase chance of hitting global optimum
def multi_start_nelder_mead(obj_func, bounds, n_starts=10):
    best_res = None
    
    for i in range(n_starts):
        # Generate a random starting point within your bounds
        x0 = np.random.uniform(bounds[:, 0], bounds[:, 1])
        
        res = minimize(obj_func, x0, method='Nelder-Mead', tol=0.01, options={'adaptive': True,'maxiter':20})
        print(str(i+1)+' out of '+str(n_starts)+' takes done')
        if best_res is None or res.fun < best_res.fun:
            best_res = res
            
    return best_res

In [ ]:
# first step is two fit size factor distribution
# let's assume skew normal distribution, which allows assymetricity

s_mle, loc_mle, scale_mle = stats.lognorm.fit(sfs['sf'],floc=0)

print([s_mle, loc_mle, scale_mle])
SF_s = np.round(s_mle,3)
SF_loc = np.round(loc_mle,3) 
SF_scale = np.round(scale_mle,3)

sf_1 = stats.lognorm.rvs(s=s_mle,loc=loc_mle, scale=scale_mle, size=1000) # _1 specifies condition 1
log2_sf_1 = np.log2(sf_1)

bins = 10
bin_edges = np.linspace(np.min(sfs['log2_sf']), np.max(sfs['log2_sf']), bins + 1)

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=True, sharex=True, figsize=(2.5, 1.2))

ax = sns.histplot(sfs['log2_sf'],stat='proportion',bins=bin_edges,color='blue',label='True SFs')
ax = sns.histplot(log2_sf_1,stat='proportion',bins=bin_edges,color='orange',label='modelled SFs')

ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=5,
)

In [ ]:
observed_Total = list(np.log2(data['CTRL_nCPM_Total']))
observed_SubFraction = list(np.log2(data['CTRL_nCPM_PD']))

# first step is to fit distribution of norm. expr values in Total 
obs_list = observed_Total
# initial_guess_1Step = [np.median(obs_list),np.std(obs_list),2,0.01] # initial guess is somewhat similar to method of moments
bounds = np.array([[1, 10], [1, 12], [0, 12]])
res = multi_start_nelder_mead(compute_KL_div_with_simulation_for_Total, bounds, n_starts=5)

# res = minimize(compute_KL_div_with_simulation_for_Total, initial_guess_1Step, method='Nelder-Mead', tol=0.01,options={'maxiter':20})
TotExpr_df, log_TotExpr_loc,log_TotExpr_scale = res.x # these are optimized values, they will be used downstream

print([TotExpr_df, log_TotExpr_loc,log_TotExpr_scale])

KL_dev_1step = compute_KL_div_with_simulation_for_Total(res.x,show_plot=True,Nsimul=10) # plot results from first step
print('1 step, KL div: '+str(KL_dev_1step))

In [ ]:
kl_div_vals_opt = compute_KL_div_with_simulation_for_Total([TotExpr_df, log_TotExpr_loc,log_TotExpr_scale],show_plot=False,Nsimul=100,return_list_of_vals=True) # values at Max Likelihood
# make draws from bounds
N_draws = 10
Total_KLdiv_vals,draw_ids,param_vals = [],[],[]
for i in range(N_draws):
    params_draw = np.random.uniform(bounds[:, 0], bounds[:, 1])
    kl_div_vals_draw = compute_KL_div_with_simulation_for_Total(params_draw,show_plot=False,Nsimul=100,return_list_of_vals=True) # values at Max Likelihood
    Total_KLdiv_vals = Total_KLdiv_vals+list(kl_div_vals_draw)
    draw_ids = draw_ids+[i]*len(kl_div_vals_draw)
    param_vals = param_vals+[params_draw]*len(kl_div_vals_draw)
    
# log2 likelihood can actually be approximated by KL divergence: https://arxiv.org/pdf/1404.2000
likelihood_opt = 2**((-1)*np.array(kl_div_vals_opt))

# calculate volume of multi-dimensional uniform distribution
dim_widths = []
for elem in bounds:
    dim_widths.append(elem[1]-elem[0])
total_volume = np.prod(dim_widths)
prior_density = 1/total_volume # this is uniform prior

numerator = likelihood_opt*prior_density
denominator = np.mean(2**((-1)*np.array(Total_KLdiv_vals))) # from the tutorial https://engineering.purdue.edu/kak/Tutorials/MonteCarloInBayesian.pdf, page 19
posterior_vals_opt = numerator/denominator

# make a dataframe for analyzing credible interval
df = pd.DataFrame([draw_ids,Total_KLdiv_vals]).transpose()
df.columns = ['draw_id','KL_div']
param_df = pd.DataFrame(param_vals)
likelihood_function_df = pd.concat([df,param_df],axis=1)
likelihood_function_df['Likelihood'] = 2**((-1)*likelihood_function_df['KL_div'])

max_LogLikelihood = np.mean((-1)*np.array(kl_div_vals_opt))
likelihood_function_df['D'] = 2*(max_LogLikelihood-((-1)*likelihood_function_df['KL_div']))  # Wilks test statistic
df_Wilks = len(bounds) # degrees of freedom is just number of parameters, because we can walk in such-dimensional space
likelihood_function_df['CI'] = stats.chi2.cdf(x=likelihood_function_df['D'],df=df_Wilks)

# we can then get the range of values for every paramter which correspond to CI under specified threshold, e.g. like that:
# likelihood_function_df.loc[likelihood_function_df['CI']<CI_thr].reset_index(drop=True) 

In [ ]:
CI_thr = 0.95
likelihood_function_df.loc[likelihood_function_df['CI']<CI_thr].reset_index(drop=True)

In [ ]:
likelihood_function_df.loc[likelihood_function_df['D']<1.92].reset_index(drop=True)

In [ ]:
sns.scatterplot(data = likelihood_function_df,x=0,y=1,hue='Likelihood')

In [ ]:
log2L = (-1)*kl_div # log2 likelihood can actually be approximated by KL divergence: https://arxiv.org/pdf/1404.2000

In [ ]:
# second step is to fit distribution of norm. expr values in sub-fraction, constrained on the Total
obs_list = observed_SubFraction

bounds = np.array([[-1, 12], [-1, 12]])
res = multi_start_nelder_mead(compute_KL_div_with_simulation_for_SubFraction, bounds, n_starts=5)

log2_f_alpha,log2_f_beta = res.x # these are optimized values, they will be used downstream

print([log2_f_alpha,log2_f_beta])

KL_dev_2step = compute_KL_div_with_simulation_for_SubFraction(res.x,show_plot=True,Nsimul=10)
print('2nd step, KL div: '+str(KL_dev_2step))

In [ ]:
sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=True, sharex=True, figsize=(2.5, 1.2))

ax = sns.histplot(stats.beta.rvs(a=1+2**log2_f_alpha, b=1+2**log2_f_beta, size=len(observed_SubFraction)),color='royalblue',stat='proportion')
ax.set(xlabel='proportion of molecules\nin a sub-fraction,\nper gene')

In [ ]:
# collect estimated parameters


In [ ]:

p = stats.beta.rvs(a=2.5, b=2.5, size=N_g) # proportion of molecules in a fraction
# p_2 = p # not changed
# p_2 = stats.beta.rvs(a=2.5, b=0.5, size=N_g) # proportion is changed randomly in condition 2

p_2_unadj = (2**(Truelog2FC*(+1.1)))*p # proportion is changed along the log2FC in condition 2
# rescale such that each proportion would be from 0 to 1
p_2 = quantile_normalize_to_reference(p_2_unadj, p)

TrueLog2FC_in_props = np.log2(p_2+0.00001)-np.log2(p+0.00001)

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 2.5))

ax = sns.scatterplot(x=p,y=p_2,s=3,alpha=0.3)
ax.set(title='proportion of mRNA molecules in a sub-fraction',xlabel='condition 1',ylabel='condition 2')

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 2.5))

ax = sns.scatterplot(x=Truelog2FC,y=TrueLog2FC_in_props,s=3,alpha=0.3)
ax.set(xlabel='True $log_2$ FC values of\nrelative mRNA expression level\nin Total',
       ylabel='True $log_2$ FC of sub-fraction proportions')

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 1.2))
ax = sns.histplot(p,stat='density',color='green',label='condition 1')
ax = sns.histplot(p_2,stat='density',color='orange',label='condition 2')
ax.set(xlabel='proportion of mRNA molecules in a sub-fraction\n(not observed)',
       ylabel='density',xlim=(-0.01,1.01))
ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=5,
)

f_1 = p*r_1 # True number of molecules in a fraction, condition 1
f_2 = p_2*r_2 # True number of molecules in a fraction, condition 2

# now, we do Poisson sampling for every bioreplicate
expr_data,cols = [],[]
total_c1_samples,total_c2_samples,frac_c1_samples,frac_c2_samples = [],[],[],[]
pseudoexpr = 0.001 # needed to make Poisson work
for repl in range(N_r):
    # true size factor is a random variable
    log2_sf_1,log2_sf_2,log2_f_sf_1,log2_f_sf_2 = stats.norm.rvs(loc=0, scale=1, size=4)
    sf_1,sf_2,f_sf_1,f_sf_2 = 2**log2_sf_1,2**log2_sf_2,2**log2_f_sf_1,2**log2_f_sf_2

    expr_data.append(stats.poisson.rvs(r_1*sf_1+pseudoexpr, loc=0, random_state=None))
    expr_data.append(stats.poisson.rvs(r_2*sf_2+pseudoexpr, loc=0, random_state=None))
    expr_data.append(stats.poisson.rvs(f_1*f_sf_1+pseudoexpr, loc=0, random_state=None))
    expr_data.append(stats.poisson.rvs(f_2*f_sf_2+pseudoexpr, loc=0, random_state=None))

    cols = cols + [str(repl)+'_c1_Total',str(repl)+'_c2_Total',str(repl)+'_c1_Frac',str(repl)+'_c2_Frac']
    total_c1_samples.append(str(repl)+'_c1_Total')
    total_c2_samples.append(str(repl)+'_c2_Total')
    frac_c1_samples.append(str(repl)+'_c1_Frac')
    frac_c2_samples.append(str(repl)+'_c2_Frac')
    
expr_df = pd.DataFrame(expr_data).transpose()
expr_df.columns = cols

# do normalization like Deseq2
expr_df['min_val'] = expr_df[cols].min(1)
tmp_df = expr_df.loc[expr_df['min_val']>0].reset_index(drop=True)
tmp_df['geom_mean'] = (tmp_df[cols].prod(1))**(1/len(cols))
tmp_df[cols] = tmp_df[cols].div(tmp_df['geom_mean'],axis=0)
sfs = tmp_df[cols].median(0)

norm_expr_df = np.log2((expr_df[cols]+1).div(sfs,axis=1)) # log2, pseudocount=1
norm_expr_df['log2FC_Total'] = norm_expr_df[total_c2_samples].mean(1)-norm_expr_df[total_c1_samples].mean(1)
norm_expr_df['log2FC_Frac'] = norm_expr_df[frac_c2_samples].mean(1)-norm_expr_df[frac_c1_samples].mean(1)

data = norm_expr_df.reset_index(drop=True).copy()

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 1.2))
ax = sns.histplot(data['0_c1_Total'],bins=30,color='grey',alpha=0.5,stat='density',
                  label='Total, condition 1')
ax = sns.histplot(data['0_c1_Frac'],bins=30,color='green',alpha=0.5,stat='density',
                  label='sub-fraction, condition 1')
ax = sns.histplot(data['0_c2_Total'],bins=30,color='royalblue',alpha=0.5,stat='density',
                  label='Total, condition 2')
ax = sns.histplot(data['0_c2_Frac'],bins=30,color='orange',alpha=0.5,stat='density',
                  label='sub-fraction, condition 2')

ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=5,
)
ax.set(xlabel='$log_2$ norm. gene expression\n(observed)',ylabel='density')


sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 2.5))

x_feature,y_feature = 'log2FC_Total','log2FC_Frac'

ax = sns.scatterplot(data = data, x=x_feature,y=y_feature,s=3,alpha=0.3)
diag_df = pd.DataFrame(
    [[ax.get_xlim()[0], ax.get_xlim()[0]], [ax.get_xlim()[1], ax.get_xlim()[1]]],
    columns=[x_feature, y_feature],
)
ax = sns.lineplot(
    data=diag_df, y=y_feature, x=x_feature, color="black", label="diagonal",
    zorder=1,
)

data["c"] = 1  # constant
X = data[["c", x_feature]].values
# X = data[[x_feature]].values
y = data[[y_feature]].values

mod = OLS(endog=y, exog=X)
reg = mod.fit()
data["y_pred"] = reg.predict(X)
ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="green", label="OLS, slope="+str(np.round(reg.params[-1],2)),zorder=1)

# Apply PCA with 1 component
pca = PCA(n_components=1)
pca.fit(data[[x_feature,y_feature]].values)
v1, v2 = pca.components_[0] # The direction of the line is given by the first eigenvector
slope_pca = v2 / v1
intercept_pca = np.mean(data[y_feature]) - slope_pca * np.mean(data[x_feature]) # The line passes through the mean of the data
data["y_pred"] = intercept_pca+slope_pca*data[x_feature]
ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="magenta", label="PC_1, slope="+str(np.round(slope_pca,2)),zorder=1)

ax.set(xlabel='$log_2$FC Total',ylabel='$log_2$FC sub-fraction',xlim=(-3,3),ylim=(-3,3))

In [ ]:
###
# Analyze the shape of the nCPM values in Total vs PD samples
# fit some parameters of our simulation model using Maximum Likelihood
###

NonCtrl_conditions = ["DMSO", "H2O2", "DTT", "Tun", "minusD", "CCCP"]

analysis_res_list = []

for NonCtrlCond in NonCtrl_conditions:
    data = correspondence_df.loc[correspondence_df["condition"] == NonCtrlCond].copy()
    CtrlCond = list(data['CTRL'].unique())
    if len(CtrlCond)>1:
        print('[WARNING] multiple CTRL conditions specified for '+NonCtrlCond)
    CtrlCond = CtrlCond[0] # take first value in any case
        
    sign_thr = 0.05

    def assign_category(x):
        if x["padj_Total"] < sign_thr:
            if x["padj_PD"] < sign_thr:
                return "both"
            else:
                return "only Total"
        else:
            if x["padj_PD"] < sign_thr:
                return "only PD"
            else:
                return "none"

    data["category"] = data.apply(lambda x: assign_category(x), 1)
    data["t"] = 1
    gr = data.groupby("category").agg({"t": np.sum}).reset_index()
    relabel_dict = {}
    for index_gr, row_gr in gr.iterrows():
        relabel_dict[row_gr["category"]] = (
            row_gr["category"]
            + ", "
            + str(np.round(row_gr["t"] / 1000, 2))
            + " K genes"
        )
    data["DE significance"] = data["category"].map(relabel_dict).astype("category")
    gr["DE significance"] = gr["category"].map(relabel_dict).astype("category")

    # data['-$log_{10}padj$'] = (-1)*np.log10((data['padj_mRNA']*data['padj_Ribo'])**0.5)
    data["-$log_{10}padj$"] = (-1) * np.log10(data[["padj_Total", "padj_PD"]].min(1))
    
    # add gene length information just in case
    data = pd.merge(res_FC[['gene_id','gene_length']],data,how='right',on=['gene_id'])

    # calculate mean values in each condition from estimated log2FC and Mean values
    data['CTRL_nCPM_Total'] = (data['baseMean_Total']+1)*2/(1+2**data['log2FoldChange_Total'])
    data['CTRL_nCPM_PD'] = (data['baseMean_PD']+1)*2/(1+2**data['log2FoldChange_PD'])
    data['EXP_nCPM_Total'] = data['CTRL_nCPM_Total']*(2**data['log2FoldChange_Total'])
    data['EXP_nCPM_PD'] = data['CTRL_nCPM_PD']*(2**data['log2FoldChange_PD'])
    
    sns.set(font_scale=1)
    sns.set_style("white")
    fig, axes = plt.subplots(1, 2, sharey=True, sharex=True, figsize=(6, 3))

    for k1,condition in enumerate(['CTRL','EXP']):
        colors,labels = ['green','orange'],['Total','PD']
        for k2,label in enumerate(labels):
            x_feauture_nonLog = condition+'_nCPM_'+label
            x_feauture = 'log2_'+x_feauture_nonLog
            data[x_feauture] = np.log2(data[x_feauture_nonLog])
            # observed
            ax = sns.histplot(
                ax=axes[k1],
                data=data,
                x=x_feauture,
                stat='density',
                color=colors[k2],
                zorder=2,
                label=label+'-obs',
                alpha=0.7,
            )
            
        ax.set(title=(CtrlCond if condition=='CTRL' else NonCtrlCond),
              xlabel='$log_2$ nCPM')
        ax.legend(
            loc="upper left",
            borderaxespad=0.0,
            title="",
            ncols=1,
            markerscale=5,
        )    
    break

### Analyze difference in FoldChanges

In [ ]:
genes_of_interest_list = [
    "ACT1",
    "PGK1",
    "BSC1",
    "DCP2",
    "MET10",
    # "TPI1",
    # "RLM1",
    # "ATP11",
    # "ILM1",
    # "MRPL38",
    # "AIM2",
    # "PTP3",
    # "NSL1",
    "RPS6A",
    "RPS9A",
    "RPS9B",
    "RPL19B",
]

NonCtrl_conditions = ["DMSO", "H2O2", "DTT", "Tun", "minusD", "CCCP"]

analysis_res_list = []

for NonCtrlCond in NonCtrl_conditions:

    sns.set(font_scale=1)
    sns.set_style("white")
    fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(5, 5))

    data = correspondence_df.loc[correspondence_df["condition"] == NonCtrlCond].copy()
    CtrlCond = list(data['CTRL'].unique())
    if len(CtrlCond)>1:
        print('[WARNING] multiple CTRL conditions specified for '+NonCtrlCond)
    CtrlCond = CtrlCond[0] # take first value in any case
        
    sign_thr = 0.05

    def assign_category(x):
        if x["padj_Total"] < sign_thr:
            if x["padj_PD"] < sign_thr:
                return "both"
            else:
                return "only Total"
        else:
            if x["padj_PD"] < sign_thr:
                return "only PD"
            else:
                return "none"

    data["category"] = data.apply(lambda x: assign_category(x), 1)
    data["t"] = 1
    gr = data.groupby("category").agg({"t": np.sum}).reset_index()
    relabel_dict = {}
    for index_gr, row_gr in gr.iterrows():
        relabel_dict[row_gr["category"]] = (
            row_gr["category"]
            + ", "
            + str(np.round(row_gr["t"] / 1000, 2))
            + " K genes"
        )
    data["DE significance"] = data["category"].map(relabel_dict).astype("category")
    gr["DE significance"] = gr["category"].map(relabel_dict).astype("category")

    # data['-$log_{10}padj$'] = (-1)*np.log10((data['padj_mRNA']*data['padj_Ribo'])**0.5)
    data["-$log_{10}padj$"] = (-1) * np.log10(data[["padj_Total", "padj_PD"]].min(1))

    x_feature, y_feature, hue = (
        "log2FoldChange_Total",
        "log2FoldChange_PD",
        "DE significance",
    )
    xlabel, ylabel = "$log_2FC$ Total", "$log_2FC$ PD"
    hue_order = list(gr.sort_values(["category"], ascending=True)[hue])
    palette = ["red", "grey", "blue", "orange"]
    alpha_param, s_param = 0.3, 3
    size_param = "-$log_{10}padj$"

    ax = sns.scatterplot(
        data=data,
        x=x_feature,
        y=y_feature,
        hue=hue,
        hue_order=hue_order,
        palette=palette,
        alpha=alpha_param,
        s=s_param,
        # size = size_param, sizes=(1, 40),
        zorder=2,
    )
    diag_df = pd.DataFrame(
        [[ax.get_xlim()[0], ax.get_xlim()[0]], [ax.get_xlim()[1], ax.get_xlim()[1]]],
        columns=[x_feature, y_feature],
    )
    ax = sns.lineplot(
        data=diag_df, y=y_feature, x=x_feature, color="black", label="diagonal",
        zorder=1,
    )

    # plot selected genes
    interesting_subset = data.loc[
        (data["gene_name"].isin(genes_of_interest_list))
    ].reset_index(drop=True)
    ax = sns.scatterplot(
        data=interesting_subset,
        x=x_feature,
        y=y_feature,
        hue=hue,
        hue_order=hue_order,
        palette=palette,
        legend=False,
        alpha=1,
        s=25,
        edgecolor="black",
        linewidth=1,
        zorder=3,
    )
    texts = []
    for index, row in interesting_subset.iterrows():
        texts.append(
            ax.annotate(
                xy=(row[x_feature], row[y_feature]),
                text=row["gene_name"],
                size=10,
                ha="left",
            )
        )

    data["c"] = 1  # constant
    # X = data[["c", x_feature]].values
    X = data[[x_feature]].values
    y = data[[y_feature]].values

    # mod = OLS(endog=y, exog=X)
    # reg = mod.fit()
    # data["y_pred"] = reg.predict(X)
    # ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="green", label="OLS",zorder=1)

    # Apply PCA with 1 component
    pca = PCA(n_components=1)
    pca.fit(data[[x_feature,y_feature]].values)
    v1, v2 = pca.components_[0] # The direction of the line is given by the first eigenvector
    slope_pca = v2 / v1
    intercept_pca = np.mean(y) - slope_pca * np.mean(X) # The line passes through the mean of the data
    data["y_pred"] = intercept_pca+slope_pca*data[x_feature]
    ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="magenta", label="PC_1",zorder=1)

    # save PCA slope per gene, for later use
    analysis_res_list.append([NonCtrlCond,CtrlCond,slope_pca])
    
    # w = data['-$log_{10}padj$'].values
    # mod = WLS(endog = y,exog = X,weights=w)
    # reg = mod.fit()
    # data['y_pred'] = reg.predict(X)
    # ax = sns.lineplot(data = data,y='y_pred',x=x_feature,color='lime',label='WLS (weigthed by padj)')

    # mod = QuantReg(endog = y,exog = X)
    # reg = mod.fit(q=0.5,max_iter=500)
    # data['y_pred'] = reg.predict(X)
    # ax = sns.lineplot(data = data,y='y_pred',x=x_feature,color='olive',label='quantile regression',zorder=1)
    
    ax.set(xlabel=xlabel, ylabel=ylabel, title=NonCtrlCond + " -vs- "+CtrlCond,
          xlim=(-2.5,2.5),ylim=(-2.5,2.5))
    %time adjust_text(texts,arrowprops=dict(arrowstyle='-', color='black'),ax=ax,)
    
    ax.tick_params(left=True, bottom=True)
    ax.legend(
        bbox_to_anchor=(1.05, 1.00),
        loc="upper left",
        borderaxespad=0.0,
        title="",
        ncols=1,
        markerscale=5,
    )

    out = subprocess.check_output(
        "mkdir -p "
        + subdirs["figures_dir"]
        + "gene_expression/DE_analysis_over_conditions/",
        shell=True,
    )
    fig.savefig(
        subdirs["figures_dir"]
        + "gene_expression/DE_analysis_over_conditions/"
        + NonCtrlCond + '_vs_' + CtrlCond
        + ".log2FC_Total_vs_log2FC_PD.png",
        bbox_inches="tight",
        dpi=300,
    )
    fig.savefig(
        subdirs["figures_dir"]
        + "gene_expression/DE_analysis_over_conditions/"
        + NonCtrlCond + '_vs_' + CtrlCond
        + ".log2FC_Total_vs_log2FC_PD.pdf",
        bbox_inches="tight",
        dpi=300,
    )

In [ ]:
# collected PCA slopes for every treatment
analysis_res_df = pd.DataFrame(analysis_res_list,columns=['condition','CTRL','slope_pca'])
analysis_res_df

In [ ]:
# Run DeltaTE-type of analysis
# Here, we analyse differential recruitement into p-bodies, instead of differential translation efficiency
# so we call the metric RE, standing for "recruitment efficiency"
# conda activate deseq

raw_matrix = res_FC.copy()  # prepare input for Deseq2
raw_matrix[samples_list] = (
    raw_matrix[samples_list] - 1
)  # we don't need pseudocount for Deseq2
raw_matrix[samples_list] = np.round(raw_matrix[samples_list]).astype("int")

list_of_output_tables = []
command = ""

exclude_replicates = []

CtrlNonCtrl_conditions = [["DMSO","CCCP"], ['UT',"DMSO"], ['UT',"DTT"], ['UT',"H2O2"], ['DMSO',"Tun"], ['UT',"minusD"]]
EXPfraction = 'PD'

for k2, CtrlNonCtrl_condition in enumerate(CtrlNonCtrl_conditions):
    CtrlCond,NonCtrlCond = CtrlNonCtrl_condition
    sub_sel_metadata = (
        sel_files.loc[
            (~sel_files["replicate"].isin(exclude_replicates))
            & (sel_files["condition"].isin([CtrlCond, NonCtrlCond]))
        ][["merged_sample", "fraction", "condition", "replicate"]]
        .reset_index(drop=True)
        .copy()
    )
    if (
        len(sub_sel_metadata) >= 4
        and len(sub_sel_metadata["condition"].unique()) == 2
    ):
        sub_sel_metadata["EXP_CTL"] = sub_sel_metadata.apply(
            lambda x: "EXP" if (x["condition"] == NonCtrlCond and x['fraction'] == EXPfraction) else "CTL", 1
        )
        index_cols = ["gene_id"]
        sub_samples_list = list(sub_sel_metadata["merged_sample"])
        cur_res = raw_matrix[index_cols + sub_samples_list].reset_index(drop=True)
        cur_res[sub_samples_list] = (
            cur_res[sub_samples_list].fillna(0).astype("int")
        )
        
        cur_res = (
            cur_res[["gene_id"] + sub_samples_list]
            .drop_duplicates("gene_id")
            .reset_index(drop=True)
        )  # to double-check that there are no duplicate entries

        prefix_path = (
            subdirs["tables_dir"]
            + "gene_expression/"
            + "delta_RE/"
            + NonCtrlCond + "_vs_" + CtrlCond
            + "/"
        )

        out = subprocess.check_output(
            "mkdir -p " + prefix_path,
            shell=True,
        )

        counts_file_path = prefix_path + "counts_deltaRE.tsv"
        metadata_file_path = prefix_path + "metadata_deltaRE.tsv"
        cur_res.to_csv(
            counts_file_path,
            sep=str("\t"),
            header=True,
            index=None,
            quoting=csv.QUOTE_NONE,
        )

        # default Deseq2 model, no confounders
        metadata_cols = ["EXP_CTL"] # condition:fraction
        sub_sel_metadata['confounder_1'] = (sub_sel_metadata['condition']==NonCtrlCond).astype('int') # condition
        sub_sel_metadata['confounder_2'] = (sub_sel_metadata['fraction']==EXPfraction).astype('int') # fraction
        sub_sel_metadata['confounder_3'] = 0 # batch, not used for now
        metadata_cols = metadata_cols+['confounder_1','confounder_2','confounder_3']

        sub_sel_metadata = sub_sel_metadata[["merged_sample"] + metadata_cols]
        sub_sel_metadata.to_csv(
            metadata_file_path,
            sep=str("\t"),
            header=True,
            index=None,
            quoting=csv.QUOTE_NONE,
        )

        outdir_path = prefix_path
        out = subprocess.check_output("mkdir -p " + outdir_path, shell=True)
        command = (
            command
            + """Rscript """
            + subdirs["wf_dir"]
            + "scripts/run_deseq2.r"
            + " "
            + counts_file_path
            + " "
            + metadata_file_path
            + " EXP "
            + outdir_path
            + " fake_gene"
            + " two_confounders"
            + " 0" # use lfc=0 as null hypothesis
            + " TRUE; " # use apeglm-shrinked log2FC values, for better visualization
        )
        list_of_output_tables.append(
            [NonCtrlCond, CtrlCond, "condition_fraction", outdir_path + "DE_table.tsv"]
        )
command

In [ ]:
deltaRE_results_links = pd.DataFrame(
    list_of_output_tables, columns=["condition",'CTRL', "confounders", "path"]
)

In [ ]:
deltaRE_results_links

In [ ]:
genes.loc[genes['gene_name'].astype('str').str.contains('|'.join(['DCP2','RPS9','RPL19B']))]

In [ ]:
genes_of_interest_list = [
    "ACT1",
    "PGK1",
    "BSC1",
    "DCP2",
    "MET10",
    # "TPI1",
    # "RLM1",
    # "ATP11",
    # "ILM1",
    # "MRPL38",
    # "AIM2",
    # "PTP3",
    # "NSL1",
    "RPS6A",
    "RPS9A",
    "RPS9B",
    "RPL19B",
]

confounders = "condition_fraction"

x_feature, y_feature, hue = "log2_baseMean", "log2FoldChange", "DE_labelled"
alpha_param, s_param = 0.3, 5
cat1 = "condition"

deseq_results = []  # list to store the Deseq output tables

deseq_results_links_cur = deltaRE_results_links.loc[
    deltaRE_results_links["confounders"] == confounders
].reset_index(drop=True)

sns.set(font_scale=0.8)
sns.set_style("white")
fig, axes = plt.subplots(
    1,
    len(deseq_results_links_cur[cat1].unique()),
    sharey=False,
    sharex=False,
    figsize=(
        4 * len(deseq_results_links_cur[cat1].unique()) ,
        4 ,
    ),
)

for i, cat1_val in enumerate(["DMSO", "H2O2", "DTT", "Tun", "minusD", "CCCP"]):
    row = deseq_results_links_cur.loc[
        (deseq_results_links_cur[cat1] == cat1_val)
    ]
    if len(row) > 0:
        row = row.iloc[0]
        CtrlCond = row['CTRL']
        if os.path.isfile(row["path"]):
            data = pd.read_csv(
                row["path"], delimiter="\t", index_col=None, header=0
            )
            data[["pvalue", "padj"]] = data[["pvalue", "padj"]].fillna(1)
            data["log2FoldChange"] = data["log2FoldChange"].fillna(0)
            data["log2_baseMean"] = np.log2(data["baseMean"] + 1)
            data["DE"] = (data["padj"] < 0.05).astype("int") * (
                (data["log2FoldChange"] > 0).astype("int") * 2 - 1
            )
            data = data.rename(columns={"gene_name": "gene_id"})

            data = pd.merge(genes, data, how="right", on="gene_id")

            # significance score
            min_positive_pval = data.loc[data["pvalue"] > 0]["pvalue"].min()
            data["DE_score"] = (
                (-1)
                * np.log10(data["pvalue"] + min_positive_pval)
                * ((data["log2FoldChange"] > 0).astype("int") * 2 - 1)
            )

            data["t"] = 1
            # add abundance of transcripts to the category labels
            gr = data.groupby(["DE"]).agg({"t": sum}).reset_index()
            short_dict = {-1: "down", 0: "NS", 1: "up"}
            relabel_dict = {}
            for index_gr, row_gr in gr.iterrows():
                relabel_dict[row_gr["DE"]] = (
                    short_dict[row_gr["DE"]]
                    + ", "
                    + str(np.round(row_gr["t"] / 1000, 2))
                    + " K genes"
                )
            data[hue] = data["DE"].map(relabel_dict).astype("category")

            data["confounding_factor"] = confounders
            data[cat1] = cat1_val
            data['CTRL'] = CtrlCond
            deseq_results.append(data.copy())

            gr[hue] = gr["DE"].map(relabel_dict).astype("category")
            gr = gr.sort_values("DE", ascending=False).reset_index(drop=True)

            hue_order = list(gr[hue])
            palette = ["red", "grey", "blue"]

            # data1 = data.loc[data['DE']!=0]
            # data2 = data.loc[data['DE']==0].sample(min(len(data1)*3,len(data)/2))
            # data_sampled = pd.concat([data1,data2]).reset_index(drop=True)
            data_sampled = data.copy()

            ### MA-plot
            ax = sns.scatterplot(
                ax=(
                    axes[i]
                    if (len(deseq_results_links_cur[cat1].unique()) > 1)
                    else axes
                ),
                data=data_sampled,
                x=x_feature,
                y=y_feature,
                hue=hue,
                hue_order=hue_order,
                palette=palette,
                alpha=alpha_param,
                s=s_param,
                legend=True,
            )

            # plot selected genes
            interesting_subset = data.loc[
                (data["gene_name"].isin(genes_of_interest_list))
            ].reset_index(drop=True)
            ax = sns.scatterplot(
                ax=(
                    axes[i]
                    if (len(deseq_results_links_cur[cat1].unique()) > 1)
                    else axes
                ),
                data=interesting_subset,
                x=x_feature,
                y=y_feature,
                hue=hue,
                hue_order=hue_order,
                palette=palette,
                legend=False,
                alpha=1,
                s=20,
                edgecolor="black",
                linewidth=2,
            )
            texts = []
            for index, row in interesting_subset.iterrows():
                texts.append(
                    ax.annotate(
                        xy=(row[x_feature], row[y_feature]),
                        text=row["gene_name"],
                        size=8,
                        ha="left",
                    )
                )
            %time adjust_text(texts,arrowprops=dict(arrowstyle='-', color='black'),ax=ax,force_text=(13,10))

            xlims = (ax.get_xlim()[0], ax.get_xlim()[1])
            ax.hlines(
                y=0,
                xmin=xlims[0],
                xmax=xlims[1],
                linestyle="--",
                color="grey",
                lw=1.5,
            )

        else:
            ax = sns.scatterplot(
                ax=axes[i],
                data=pd.DataFrame([[0, 0]], columns=[x_feature, y_feature]),
                x=x_feature,
                y=y_feature,
                color="white",
                alpha=0,
                s=s_param,
            )
            ax.annotate(
                xy=(0, 0),
                text="DE test is not allowed",
                size=10,
                ha="left",
                va="center",
            )
    else:
        CtrlCond = 'NA'
        ax = sns.scatterplot(
            ax=axes[i],
            data=pd.DataFrame([[0, 0]], columns=[x_feature, y_feature]),
            x=x_feature,
            y=y_feature,
            color="white",
            alpha=0,
            s=s_param,
        )
        ax.annotate(
            xy=(0, 0),
            text="experiment not performed",
            size=10,
            ha="left",
            va="center",
        )
    ax.set(title=cat1_val+'-vs-'+CtrlCond)
    ax.tick_params(left=True, bottom=True)

    if i == 0:
        ax.set(ylabel=r"$log_2$FC of RE")
    else:
        ax.set(ylabel="")
    ax.legend(
        bbox_to_anchor=(0.0, 0.01),
        loc=3,
        borderaxespad=0.0,
        title="delta RE test:",
        ncols=1,
        markerscale=2,
    )
    ax.set(xlim=(-1, 20), ylim=(-5, 5), xlabel="$log_2$ av. gene expr.")

out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "gene_expression/",
    shell=True,
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/deltaRE_analysis.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "gene_expression/deltaRE_analysis.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
All_deseq_df.loc[All_deseq_df['gene_name']=='MET10']

In [ ]:
All_deseq_df = pd.concat(deseq_results).reset_index(drop=True)

index_cols = ["gene_id", "gene_name", "gene_type", "confounding_factor", "condition",'CTRL']
feature_cols = ["baseMean","log2FoldChange", 'lfcSE', "padj",'DE_score']

rename_dict = {}
for elem in feature_cols:
    rename_dict[elem] = elem + "_RE"
deltaRE_df = All_deseq_df[index_cols+feature_cols].rename(columns=rename_dict).reset_index(drop=True)

In [ ]:
# make heatmap for comparison of stresses in terms of deltaRE
import matplotlib.patches as mpatches

feature = 'log2FoldChange_RE'

padj_tres = 0.05

data = deltaRE_df.copy()
data['comparison'] = data['condition']+'-vs-'+data['CTRL']
comparisons = list(data['comparison'].unique())

selected_genes = list(data.loc[(data['padj_RE']<padj_tres)]['gene_id'].unique()) # select genes significant in at least one stress
print(len(selected_genes))

pivot_data = pd.pivot_table(data = data,values=feature,index=['gene_id','gene_name','gene_type'],
                            columns='comparison',fill_value=0).reset_index()
pivot_data = pivot_data.loc[pivot_data['gene_id'].isin(selected_genes)].reset_index()

# make colored groups
group_color_feature = 'gene_type'
l = list(pivot_data[group_color_feature].unique())
palette = sns.color_palette("tab10", len(l))
color_dict,handles = {},[]
k = 0
for elem in l:
    color_dict[elem] = palette[k]
    handles.append(mpatches.Patch(color=palette[k], label=elem))
    k=k+1
pivot_data[group_color_feature] = pivot_data[group_color_feature].map(color_dict)

sel_cols = ['CCCP-vs-DMSO', 'DTT-vs-UT', 'H2O2-vs-UT', 'Tun-vs-DMSO', 'minusD-vs-UT']

sns.set(font_scale=1)
sns.set_style("white")

g = sns.clustermap(data=pivot_data[sel_cols],row_cluster=True, col_cluster=True,
                   metric = 'correlation',method='average',
                    # row_colors=pivot_data[[group_color_feature]],
                    yticklabels=False, xticklabels=True,
                    cbar_pos=(1.1, 0.33, .03, 0.5),
                   cbar_kws={'label':r"$log_2$FC of RE"},
                    vmin=-5, vmax=5,center=0,
                   cmap=sns.diverging_palette(220, 20, as_cmap=True),
                    figsize=(5,5),dendrogram_ratio=[0.2,0.2])
ax = g.ax_heatmap
ax.set(xlabel='',ylabel=str(len(selected_genes))+' genes '+'\n'+'with $p_{adj}$<'+str(padj_tres)+'\n'+'in at least one stress')
# ax.legend(handles=handles,bbox_to_anchor=(1.3, 0.5),loc='lower left',borderaxespad=0.0,
#           title=group_color_feature,
#           ncols=1,fontsize=10)
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "gene_expression/",
    shell=True,
)
g.savefig(
    subdirs["figures_dir"] + "gene_expression/deltaRE_heatmap.png",
    bbox_inches="tight",
    dpi=300,
)
g.savefig(
    subdirs["figures_dir"] + "gene_expression/deltaRE_heatmap.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
# output to tables: TO DO

In [ ]:
list(correspondence_df['comparison'].unique())

In [ ]:
# make nice output table and sort it
# we want to estimate roughly general PD enrichment and context-specific enrichment

correspondence_df['comparison'] = correspondence_df['condition']+'-vs-'+correspondence_df['CTRL']
correspondence_df['sorting_feature'] = correspondence_df['DE_score_PD']-correspondence_df['DE_score_Total']
# correspondence_df['sorting_feature'] = correspondence_df['log2FoldChange_PD']-correspondence_df['log2FoldChange_Total']
feature_cols = ['log2FoldChange_Total','padj_Total','log2FoldChange_PD','padj_PD','sorting_feature']
index_cols = ['gene_id','gene_name','gene_type']

sorting_features = []
for k,comparison in enumerate(list(correspondence_df['comparison'].unique())):
    tmp = correspondence_df.loc[correspondence_df['comparison']==comparison][index_cols+feature_cols].copy()
    rename_dict = {}
    for feature_col in feature_cols:
        rename_dict[feature_col] = feature_col+';'+comparison
    tmp = tmp.rename(columns = rename_dict)
    sorting_features.append('sorting_feature;'+comparison)
    if k==0:
        out_df = tmp.copy()
    else:
        out_df = pd.merge(out_df,tmp,how='left',on=index_cols)

sel_sorting_features = [
    'sorting_feature;H2O2-vs-UT',
    'sorting_feature;DTT-vs-UT',
    'sorting_feature;Tun-vs-DMSO',
    'sorting_feature;minusD-vs-UT',
    'sorting_feature;CCCP-vs-DMSO'
]

# out_df['qPDenrichment_Score'] = out_df[sel_sorting_features].quantile(axis=1,q=0.75)
out_df['meanPDenrichment_Score'] = out_df[sel_sorting_features].mean(1)
out_df['maxPDenrichment_Score'] = out_df[sel_sorting_features].max(1)
# out_df['minPDenrichment_Score'] = out_df[sel_sorting_features].min(1)
out_df = out_df.sort_values('meanPDenrichment_Score',ascending=False).reset_index(drop=True)

In [ ]:
genes.loc[genes['gene_name'].astype('str').str.contains('RPS6')]

In [ ]:
genes_of_interest_list = [
    "RPS6A",
    "ACT1",
    "PGK1",
    "BSC1",
    "TPI1",
    "RLM1",
    "ATP11",
    "ILM1",
    "MRPL38",
    "AIM2",
    "PTP3",
    "NSL1",
]

x_feature, y_feature = (
    "meanPDenrichment_Score",
    "maxPDenrichment_Score",
)
xlabel, ylabel = "$log_2FC$ Total", "$log_2FC$ PD"

data = out_df.copy()

g = sns.jointplot(data = data,x=x_feature,y=y_feature,alpha=0.5, s=3, height=5)
ax = g.ax_joint

# plot selected genes
interesting_subset = data.loc[
    (data["gene_name"].isin(genes_of_interest_list))
].reset_index(drop=True)
ax = sns.scatterplot(
    data=interesting_subset,
    x=x_feature,
    y=y_feature,
    legend=False,
    color='royalblue',
    alpha=1,
    s=25,
    edgecolor="black",
    linewidth=1,
    zorder=10,
)
texts = []
for index, row in interesting_subset.iterrows():
    texts.append(
        ax.annotate(
            xy=(row[x_feature], row[y_feature]),
            text=row["gene_name"],
            size=10,
            ha="left",
        )
    )
# ax.set(xlim=(-2,2),ylim=(-1,5))
ax.set(xlim=(-25,25),ylim=(-5,50))
ax.tick_params(left=True, bottom=True)
%time adjust_text(texts,arrowprops=dict(arrowstyle='-', color='black'),ax=ax,)

ax.set(xlabel='treatment-mean PD enrichment',ylabel='treatment-max PD enrichment')
out = subprocess.check_output(
    "mkdir -p "
    + subdirs["figures_dir"]
    + "gene_expression/",
    shell=True,
)
g.savefig(
    subdirs["figures_dir"]
    + "gene_expression/"
    + "General_vs_ContextDependent_PD_enrichment.png",
    bbox_inches="tight",
    dpi=300,
)
g.savefig(
    subdirs["figures_dir"]
    + "gene_expression/"
    + "General_vs_ContextDependent_PD_enrichment.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/DiffExpressionAnalysis.tsv"
)
out_df.to_csv(
    out_file_path, sep=str("\t"), header=True, index=None, quoting=csv.QUOTE_NONE
)

In [ ]:
out = subprocess.check_output(
    "mkdir -p "
    + subdirs["tables_dir"]
    + "gene_expression/enrichmentAnalysis/",
    shell=True,
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/enrichmentAnalysis/background.txt"
)
out_df[
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/enrichmentAnalysis/top500GeneralEnrichment.txt"
)
out_df.head(500)[
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

In [ ]:
experiment = "no_confounders"
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/comparison_with_old.Supernatant.mRNA.tsv"
)
comparison_data.to_csv(
    out_file_path, sep=str("\t"), header=True, index=None, quoting=csv.QUOTE_NONE
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/new.log2FC.Supernatant.mRNA.tsv"
)
comparison_data[["gene_id", "log2FoldChange"]].to_csv(
    out_file_path, sep=str("\t"), header=False, index=None, quoting=csv.QUOTE_NONE
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/old.log2FC.Supernatant.mRNA.tsv"
)
comparison_data[["gene_id", "log2FoldChange_old"]].to_csv(
    out_file_path, sep=str("\t"), header=False, index=None, quoting=csv.QUOTE_NONE
)

In [ ]:
(
    "mkdir -p "
    + subdirs["figures_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/"
)

In [ ]:
out = subprocess.check_output(
    "mkdir -p "
    + subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/",
    shell=True,
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/new.DE_up.tsv"
)
comparison_data.loc[comparison_data["DE"] == 1][
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/new.DE_NS.tsv"
)
comparison_data.loc[comparison_data["DE"] == 0][
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/new.DE_down.tsv"
)
comparison_data.loc[comparison_data["DE"] == -1][
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/old.DE_up.tsv"
)
comparison_data.loc[comparison_data["DE_old"] == 1][
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/old.DE_NS.tsv"
)
comparison_data.loc[comparison_data["DE_old"] == 0][
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/old.DE_down.tsv"
)
comparison_data.loc[comparison_data["DE_old"] == -1][
    ["gene_id"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis/all_genes.tsv"
)
comparison_data[["gene_id"]].drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

In [ ]:
comparison_data["t"] = 1
gr = comparison_data.groupby(["DE", "DE_old"]).agg({"t": sum}).reset_index()
gr["DE"] = gr["DE"].map({-1: "down", 1: "up", 0: "NS"})
gr["DE_old"] = gr["DE_old"].map({-1: "down", 1: "up", 0: "NS"})
gr

# Simulations

In [ ]:
# fit linear models
from statsmodels.regression.linear_model import OLS
from statsmodels.regression.linear_model import WLS
from statsmodels.regression.quantile_regression import QuantReg

In [ ]:
def quantile_normalize_to_reference(target, reference):
    """
    Normalizes 'target' so it matches the distribution of 'reference'.
    """
    # 1. Sort the reference to get the desired distribution curve
    sorted_reference = np.sort(reference)
    
    # 2. Get the rank of the target values
    # argsort twice gives the rank (0, 1, ... N-1) of each element
    rank_indices = np.argsort(np.argsort(target))
    
    # 3. Calculate percentiles (0.0 to 1.0) for the target ranks
    # Use (N - 1) to ensure the range covers 0.0 to 1.0 inclusive
    target_percentiles = rank_indices / (len(target) - 1)
    
    # 4. Calculate percentiles for the reference data
    reference_percentiles = np.linspace(0, 1, len(reference))
    
    # 5. Interpolate
    # Find what value in the sorted reference corresponds to the target's percentile
    normalized_target = np.interp(target_percentiles, reference_percentiles, sorted_reference)
    
    return normalized_target

# --- Example Usage ---

# Source data (e.g., raw gene counts or expression values)
data_source = stats.norm.rvs(loc=3, scale=2, size=1000,)

# Reference data (e.g., a "clean" normal distribution or a reference sample)
# Note: Lengths can be different!
data_reference = stats.norm.rvs(loc=0, scale=1, size=1000,)

normalized = quantile_normalize_to_reference(data_source, data_reference)

ax = sns.histplot(data_reference,color='blue')
ax = sns.histplot(data_source,color='orange')
ax = sns.histplot(normalized,color='red')

In [ ]:
def quantile_normalize_to_reference(target, reference):
    """
    Normalizes 'target' so it matches the distribution of 'reference'.
    """
    # 1. Sort the reference to get the desired distribution curve
    sorted_reference = np.sort(reference)
    
    # 2. Get the rank of the target values
    # argsort twice gives the rank (0, 1, ... N-1) of each element
    rank_indices = np.argsort(np.argsort(target))
    
    # 3. Calculate percentiles (0.0 to 1.0) for the target ranks
    # Use (N - 1) to ensure the range covers 0.0 to 1.0 inclusive
    target_percentiles = rank_indices / (len(target) - 1)
    
    # 4. Calculate percentiles for the reference data
    reference_percentiles = np.linspace(0, 1, len(reference))
    
    # 5. Interpolate
    # Find what value in the sorted reference corresponds to the target's percentile
    normalized_target = np.interp(target_percentiles, reference_percentiles, sorted_reference)
    
    return normalized_target

In [ ]:
N_r = 3 # number of replicates per condition
N_g = 7000 # number of genes

log2_r_1 = stats.norm.rvs(loc=5, scale=3, size=N_g,) # log2 molecule counts, condition 1
Abundancelog2FC = stats.norm.rvs(loc=0, scale=1.1, size=N_g,) # change in abundance is smaller in magnitude than the original abundance
log2_r_2 = Abundancelog2FC+log2_r_1 # this now has increased variance
# log2_r_2 = (log2_r_2-np.mean(log2_r_2))/(np.std(log2_r_2)/np.std(log2_r_1)) # make sure std is the same
# log2_r_2 = log2_r_2+np.mean(log2_r_1) # make sure mean is the same
log2_r_2 = quantile_normalize_to_reference(log2_r_2, log2_r_1) # make sure distribution is the same

r_1 = np.round(2**log2_r_1,0)+1 # actual count of molecules, condition 1, need at least one molecule per gene
r_2 = np.round(2**log2_r_2,0)+1 # actual count of molecules, condition 2, need at least one molecule per gene

Truelog2FC = np.log2(r_2/np.sum(r_2))-np.log2(r_1/np.sum(r_1))

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 1.2))
ax = sns.histplot(Truelog2FC,stat='density',color='grey')
ax.set(xlabel='True $log_2$ FC values of\nrelative mRNA expression level\nin Total')

p = stats.beta.rvs(a=2.5, b=2.5, size=N_g) # proportion of molecules in a fraction
# p_2 = p # not changed
# p_2 = stats.beta.rvs(a=2.5, b=0.5, size=N_g) # proportion is changed randomly in condition 2

p_2_unadj = (2**(Truelog2FC*(+1.1)))*p # proportion is changed along the log2FC in condition 2
# rescale such that each proportion would be from 0 to 1
p_2 = quantile_normalize_to_reference(p_2_unadj, p)

TrueLog2FC_in_props = np.log2(p_2+0.00001)-np.log2(p+0.00001)

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 2.5))

ax = sns.scatterplot(x=p,y=p_2,s=3,alpha=0.3)
ax.set(title='proportion of mRNA molecules in a sub-fraction',xlabel='condition 1',ylabel='condition 2')

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 2.5))

ax = sns.scatterplot(x=Truelog2FC,y=TrueLog2FC_in_props,s=3,alpha=0.3)
ax.set(xlabel='True $log_2$ FC values of\nrelative mRNA expression level\nin Total',
       ylabel='True $log_2$ FC of sub-fraction proportions')

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 1.2))
ax = sns.histplot(p,stat='density',color='green',label='condition 1')
ax = sns.histplot(p_2,stat='density',color='orange',label='condition 2')
ax.set(xlabel='proportion of mRNA molecules in a sub-fraction\n(not observed)',
       ylabel='density',xlim=(-0.01,1.01))
ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=5,
)

f_1 = p*r_1 # True number of molecules in a fraction, condition 1
f_2 = p_2*r_2 # True number of molecules in a fraction, condition 2

# now, we do Poisson sampling for every bioreplicate
expr_data,cols = [],[]
total_c1_samples,total_c2_samples,frac_c1_samples,frac_c2_samples = [],[],[],[]
pseudoexpr = 0.001 # needed to make Poisson work
for repl in range(N_r):
    # true size factor is a random variable
    log2_sf_1,log2_sf_2,log2_f_sf_1,log2_f_sf_2 = stats.norm.rvs(loc=0, scale=1, size=4)
    sf_1,sf_2,f_sf_1,f_sf_2 = 2**log2_sf_1,2**log2_sf_2,2**log2_f_sf_1,2**log2_f_sf_2

    expr_data.append(stats.poisson.rvs(r_1*sf_1+pseudoexpr, loc=0, random_state=None))
    expr_data.append(stats.poisson.rvs(r_2*sf_2+pseudoexpr, loc=0, random_state=None))
    expr_data.append(stats.poisson.rvs(f_1*f_sf_1+pseudoexpr, loc=0, random_state=None))
    expr_data.append(stats.poisson.rvs(f_2*f_sf_2+pseudoexpr, loc=0, random_state=None))

    cols = cols + [str(repl)+'_c1_Total',str(repl)+'_c2_Total',str(repl)+'_c1_Frac',str(repl)+'_c2_Frac']
    total_c1_samples.append(str(repl)+'_c1_Total')
    total_c2_samples.append(str(repl)+'_c2_Total')
    frac_c1_samples.append(str(repl)+'_c1_Frac')
    frac_c2_samples.append(str(repl)+'_c2_Frac')
    
expr_df = pd.DataFrame(expr_data).transpose()
expr_df.columns = cols

# do normalization like Deseq2
expr_df['min_val'] = expr_df[cols].min(1)
tmp_df = expr_df.loc[expr_df['min_val']>0].reset_index(drop=True)
tmp_df['geom_mean'] = (tmp_df[cols].prod(1))**(1/len(cols))
tmp_df[cols] = tmp_df[cols].div(tmp_df['geom_mean'],axis=0)
sfs = tmp_df[cols].median(0)

norm_expr_df = np.log2((expr_df[cols]+1).div(sfs,axis=1)) # log2, pseudocount=1
norm_expr_df['log2FC_Total'] = norm_expr_df[total_c2_samples].mean(1)-norm_expr_df[total_c1_samples].mean(1)
norm_expr_df['log2FC_Frac'] = norm_expr_df[frac_c2_samples].mean(1)-norm_expr_df[frac_c1_samples].mean(1)

data = norm_expr_df.reset_index(drop=True).copy()

sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 1.2))
ax = sns.histplot(data['0_c1_Total'],bins=30,color='grey',alpha=0.5,stat='density',
                  label='Total, condition 1')
ax = sns.histplot(data['0_c1_Frac'],bins=30,color='green',alpha=0.5,stat='density',
                  label='sub-fraction, condition 1')
ax = sns.histplot(data['0_c2_Total'],bins=30,color='royalblue',alpha=0.5,stat='density',
                  label='Total, condition 2')
ax = sns.histplot(data['0_c2_Frac'],bins=30,color='orange',alpha=0.5,stat='density',
                  label='sub-fraction, condition 2')

ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=5,
)
ax.set(xlabel='$log_2$ norm. gene expression\n(observed)',ylabel='density')


sns.set(font_scale=0.7)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(2.5, 2.5))

x_feature,y_feature = 'log2FC_Total','log2FC_Frac'

ax = sns.scatterplot(data = data, x=x_feature,y=y_feature,s=3,alpha=0.3)
diag_df = pd.DataFrame(
    [[ax.get_xlim()[0], ax.get_xlim()[0]], [ax.get_xlim()[1], ax.get_xlim()[1]]],
    columns=[x_feature, y_feature],
)
ax = sns.lineplot(
    data=diag_df, y=y_feature, x=x_feature, color="black", label="diagonal",
    zorder=1,
)

data["c"] = 1  # constant
X = data[["c", x_feature]].values
# X = data[[x_feature]].values
y = data[[y_feature]].values

mod = OLS(endog=y, exog=X)
reg = mod.fit()
data["y_pred"] = reg.predict(X)
ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="green", label="OLS, slope="+str(np.round(reg.params[-1],2)),zorder=1)

# Apply PCA with 1 component
pca = PCA(n_components=1)
pca.fit(data[[x_feature,y_feature]].values)
v1, v2 = pca.components_[0] # The direction of the line is given by the first eigenvector
slope_pca = v2 / v1
intercept_pca = np.mean(data[y_feature]) - slope_pca * np.mean(data[x_feature]) # The line passes through the mean of the data
data["y_pred"] = intercept_pca+slope_pca*data[x_feature]
ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="magenta", label="PC_1, slope="+str(np.round(slope_pca,2)),zorder=1)

ax.set(xlabel='$log_2$FC Total',ylabel='$log_2$FC sub-fraction',xlim=(-3,3),ylim=(-3,3))

In [ ]:
import random

bins = 50
min_value = -20
log_x = stats.t.rvs(df=100,loc=2,scale=0.3,size=1000)
x = 2**log_x
x = (x-np.median(x))*(-1) # mimics ground truth
x = x[x>min_value] # for better plotting

log_y = stats.t.rvs(df=2,loc=2,scale=0.5,size=1000)
y = 2**log_y
y = (y-np.quantile(y,q=0.5))*(-1) # mimics observed values
y = y[y>min_value] # for better plotting

# now, perform filtering from Y with weights coming from X
combined = np.concatenate([x, y])
bin_edges = np.linspace(np.min(combined), np.max(combined), bins + 1)

# calculate centers of bins
center_points = []
for k,elem in enumerate(bin_edges[:-1]):
    center_points.append(np.mean([elem,bin_edges[k+1]]))
center_points = np.array(center_points)

# Calculate histograms (densities)
x_hist, _ = np.histogram(x, bins=bin_edges, density=True)
y_hist, _ = np.histogram(y, bins=bin_edges, density=True)
# Avoid zeros
x_hist += 1e-10
y_hist += 1e-10
# Normalize to ensure sum is 1
x_hist /= x_hist.sum()
y_hist /= y_hist.sum()
# weights = x_hist/(np.max(x_hist)) # THIS IS HOW WE DEFINE WEIGHTS
weights = x_hist/(y_hist) # Instead, we need to normalize relative to frequency in observed!
weights = weights/np.max(weights)

y_subsampled = np.array([])
for k,elem in enumerate(bin_edges[:-1]):
    if k<(len(bin_edges)-1):
        y_vals_in_bin = y[(y>=elem)&(y<bin_edges[k+1])]
        number_to_subsample = int(np.round(len(y_vals_in_bin)*weights[k]))
        y_bin_sampled = np.random.choice(y_vals_in_bin,number_to_subsample,replace=False)
        y_subsampled = np.concatenate((y_subsampled, y_bin_sampled), axis=0)

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(2, 1, sharey=True, sharex=True, figsize=(6, 6))

ax = sns.histplot(ax=axes[0],x=x,stat='proportion', bins=bin_edges,color='blue',label='ground truth',element="poly",fill=False)
ax = sns.histplot(ax=axes[0],x=y,stat='proportion', bins=bin_edges,color='green',label='observed',element="poly",fill=False)
ax = sns.histplot(ax=axes[0],x=y_subsampled,stat='proportion', bins=bin_edges,color='orange',label='observed_subsampled',element="poly",fill=True,alpha=0.5)
ax.legend(
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)
ax.tick_params(left=True, bottom=True)

ax = sns.lineplot(ax=axes[1],x=center_points,y=weights,color='red',label='weights')
ax.set(ylabel='weight',xlabel='value')
ax.tick_params(left=True, bottom=True)
ax.legend(
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)

In [ ]:
import random

bins = 20
min_value = -20
log_x = stats.t.rvs(df=100,loc=2,scale=0.3,size=1000)
x = 2**log_x
x = (x-np.median(x))*(-1) # mimics ground truth
x = x[x>min_value] # for better plotting

log_y = stats.t.rvs(df=2,loc=2,scale=0.5,size=1000)
y = 2**log_y
y = (y-np.quantile(y,q=0.5))*(-1) # mimics observed values
y = y[y>min_value] # for better plotting

# now, perform filtering from Y with weights coming from X
combined = np.concatenate([x, y])
bin_edges = np.linspace(np.min(combined), np.max(combined), bins + 1)

# calculate centers of bins
center_points = []
for k,elem in enumerate(bin_edges[:-1]):
    center_points.append(np.mean([elem,bin_edges[k+1]]))
center_points = np.array(center_points)

# Calculate histograms and make CDF!
x_hist, _ = np.histogram(x, bins=bin_edges, density=True)
y_hist, _ = np.histogram(y, bins=bin_edges, density=True)
# Avoid zeros
x_hist += 1e-10
y_hist += 1e-10
# Normalize to ensure sum is 1
x_hist /= x_hist.sum()
y_hist /= y_hist.sum()

# get eCDFs
x_eCDF,cumul = [],0
for k,elem in enumerate(x_hist):
    cumul = cumul+elem
    x_eCDF.append(cumul)
x_eCDF = np.array(x_eCDF)

y_eCDF,cumul = [],0
for k,elem in enumerate(y_hist):
    cumul = cumul+elem
    y_eCDF.append(cumul)
y_eCDF = np.array(y_eCDF)

# weights = x_hist/(np.max(x_hist)) # THIS IS HOW WE DEFINE WEIGHTS

# weights = x_hist/(y_hist) # Instead, we need to normalize relative to frequency in observed!
# weights = weights/np.max(weights)

weights = x_eCDF/(y_eCDF) # Instead, we need to normalize relative to frequency in observed!
weights = weights/np.max(weights)

y_subsampled = np.array([])
for k,elem in enumerate(bin_edges[:-1]):
    if k<(len(bin_edges)-1):
        y_vals_in_bin = y[(y>=elem)&(y<bin_edges[k+1])]
        number_to_subsample = int(np.round(len(y_vals_in_bin)*weights[k]))
        y_bin_sampled = np.random.choice(y_vals_in_bin,number_to_subsample,replace=False)
        y_subsampled = np.concatenate((y_subsampled, y_bin_sampled), axis=0)

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(2, 1, sharey=True, sharex=True, figsize=(6, 6))

ax = sns.histplot(ax=axes[0],x=x,stat='proportion', bins=bin_edges,color='blue',label='ground truth',element="poly",fill=False)
ax = sns.histplot(ax=axes[0],x=y,stat='proportion', bins=bin_edges,color='green',label='observed',element="poly",fill=False)
ax = sns.histplot(ax=axes[0],x=y_subsampled,stat='proportion', bins=bin_edges,color='orange',label='observed_subsampled',element="poly",fill=True,alpha=0.5)
ax.legend(
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)
ax.tick_params(left=True, bottom=True)

ax = sns.lineplot(ax=axes[1],x=center_points,y=weights,color='red',label='weights')
ax.set(ylabel='weight',xlabel='value')
ax.tick_params(left=True, bottom=True)
ax.legend(
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)

In [ ]:
all_dG_vals = pd.read_csv(
    subdirs["main_project_dir"] + "all_10mers_RIsearch2.tsv",
    delimiter="\t",
    index_col=None,
    header=0,
)
dG_cols = all_dG_vals.columns[2:]
all_dG_vals['dG'] = all_dG_vals[dG_cols].min(1)

In [ ]:
dG_TrueCS_vals = pd.read_csv(
    subdirs["main_project_dir"] + "atlas_min69_RIsearch2_merged_constraint.tsv",
    delimiter="\t",
    index_col=None,
    header=0,
)
dG_cols = dG_TrueCS_vals.columns[2:]
dG_TrueCS_vals['dG'] = dG_TrueCS_vals[dG_cols].min(1)

In [ ]:
av_transcript_length = 24*10**3
N_transcripts = 13000 # we mean here different unique transcript isoforms
transcriptome_length = av_transcript_length*N_transcripts
av_N_of_CSs_per_PAS = 4
N_of_True_PASs = len(dG_TrueCS_vals)
N_of_True_CSs = N_of_True_PASs*av_N_of_CSs_per_PAS

P_true_CSs = N_of_True_CSs/transcriptome_length # P(true CS)
print(P_true_CSs)
true_CSs = stats.uniform.rvs(0,transcriptome_length,size=N_of_True_CSs) # randomly distribute true CSs in the genome

bins = 10
x = np.concatenate((all_dG_vals['dG'],dG_TrueCS_vals['dG']),axis=0)
bin_edges = np.linspace(np.min(x), np.max(x), bins + 1)

# calculate centers of bins
center_points = []
for k,elem in enumerate(bin_edges[:-1]):
    center_points.append(np.mean([elem,bin_edges[k+1]]))
center_points = np.array(center_points)

# Calculate histograms (densities)
hist_all, _ = np.histogram(all_dG_vals['dG'], bins=bin_edges, density=True)
hist_True, _ = np.histogram(dG_TrueCS_vals['dG'], bins=bin_edges, density=True)
# Avoid zeros
hist_all += 1e-10
hist_True += 1e-10
# Normalize to ensure sum is 1
hist_all /= hist_all.sum()
hist_True /= hist_True.sum()

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(2, 1, sharey=False, sharex=True, figsize=(6, 6))

P_dG = hist_all # P(dG)
# ax = sns.histplot(ax=axes[0],x=all_dG_vals['dG'],stat='proportion', bins=bin_edges,color='blue',label='dG histogram, randomly sampled',element="poly",fill=True,alpha=0.3)
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_dG,color='red',label='P(dG)')

P_dG_True = hist_True
# ax = sns.histplot(ax=axes[0],x=dG_TrueCS_vals['dG'],stat='proportion', bins=bin_edges,color='green',label='dG histogram, True CSs',element="poly",fill=True,alpha=0.3)
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_dG_True,color='orange',label='P(dG | True CS)')

P_True_given_dG = P_dG_True * P_true_CSs / P_dG # P(true CS|dG) - desired probability
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_True_given_dG,color='royalblue',label='P(true CS | dG)')

ax.legend(
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)

ax.set(xlabel='dG')

# ratio = P_dG_True/P_dG # ratio relative to background
ratio = P_True_given_dG/P_true_CSs # ratio relative to background
ax = sns.lineplot(ax=axes[1],x=center_points,y=ratio,color='green',label='P(true CS | dG ) / P (true CS)')
ax = sns.lineplot(ax=axes[1],x=center_points,y=np.zeros(len(ratio))+1,color='black')

In [ ]:
dG_TrueCS_vals = pd.read_csv(
    subdirs["main_project_dir"] + "atlas_min69_RIsearch2_merged_constraint.tsv",
    delimiter="\t",
    index_col=None,
    header=0,
)
dG_cols = dG_TrueCS_vals.columns[2:]
dG_TrueCS_vals['dG'] = dG_TrueCS_vals[dG_cols].min(1)

In [ ]:
IP_dG_vals = stats.t.rvs(df=3,loc=-5,scale=0.1,size=1000) # IP simulation
IP_dG_vals.min(),IP_dG_vals.max()

In [ ]:
av_N_of_CSs_per_PAS = 4
N_of_True_PASs = len(dG_TrueCS_vals)
N_of_True_CSs = N_of_True_PASs*av_N_of_CSs_per_PAS

N_of_measured_CSs = 600000/13*17

P_true_CSs = N_of_True_CSs/N_of_measured_CSs # P(true CS)
# P_true_CSs = 0.3
print(P_true_CSs)

IP_dG_vals = stats.t.rvs(df=3,loc=-5,scale=0.3,size=1000) # IP simulation

bins = 30
x = np.concatenate((IP_dG_vals,dG_TrueCS_vals['dG']),axis=0)
bin_edges = np.linspace(np.min(x), np.max(x), bins + 1)

# calculate centers of bins
center_points = []
for k,elem in enumerate(bin_edges[:-1]):
    center_points.append(np.mean([elem,bin_edges[k+1]]))
center_points = np.array(center_points)

# Calculate histograms (densities)
hist_IP, _ = np.histogram(IP_dG_vals, bins=bin_edges, density=True)
hist_True, _ = np.histogram(dG_TrueCS_vals['dG'], bins=bin_edges, density=True)
# Avoid zeros
hist_IP += 1e-10
hist_True += 1e-10
# Normalize to ensure sum is 1
hist_IP /= hist_IP.sum()
hist_True /= hist_True.sum()

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(2, 1, sharey=False, sharex=True, figsize=(6, 6))

P_dG_True = hist_True
# ax = sns.histplot(ax=axes[0],x=dG_TrueCS_vals['dG'],stat='proportion', bins=bin_edges,color='green',label='dG histogram, True CSs',element="poly",fill=True,alpha=0.3)
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_dG_True,color='orange',label='P(dG | True CS)')

P_dG_IP = hist_IP
# ax = sns.histplot(ax=axes[0],x=dG_TrueCS_vals['dG'],stat='proportion', bins=bin_edges,color='green',label='dG histogram, True CSs',element="poly",fill=True,alpha=0.3)
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_dG_IP,color='green',label='P(dG | IP CS)')

P_dG = hist_True*P_true_CSs + hist_IP*(1-P_true_CSs) # P(dG)=P(dG|T)*P(T)+P(dG|IP)*P(IP)
# ax = sns.histplot(ax=axes[0],x=all_dG_vals['dG'],stat='proportion', bins=bin_edges,color='blue',label='dG histogram, randomly sampled',element="poly",fill=True,alpha=0.3)
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_dG,color='red',label='P(dG)')

P_True_given_dG = P_dG_True * P_true_CSs / P_dG # P(true CS|dG) - desired probability
ax = sns.lineplot(ax=axes[0],x=center_points,y=P_True_given_dG,color='royalblue',label='P(true CS | dG)')

ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)

# ratio = P_dG_True/P_dG # ratio relative to background
ratio = P_True_given_dG/P_true_CSs # ratio relative to background
ax = sns.lineplot(ax=axes[1],x=center_points,y=ratio,color='black',label='P(true CS | dG ) / P (true CS)')
ax = sns.lineplot(ax=axes[1],x=center_points,y=np.zeros(len(ratio))+1,color='grey')
ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)
ax.set(xlabel='dG')

# Translation efficiency analysis

In [ ]:
mRNA_counts_path = (
    subdirs["tables_dir"] + "gene_expression/genomic_mapping_UM/mRNA/S/counts.tsv"
)
Ribo_counts_path = (
    subdirs["tables_dir"] + "gene_expression/genomic_mapping_UM/Ribo/S/counts.tsv"
)

mRNA_metadata_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/mRNA/S/metadata.no_confounders.tsv"
)
Ribo_metadata_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/Ribo/S/metadata.no_confounders.tsv"
)

mRNA_deseqOut_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/mRNA/S/no_confounders/DE_table.tsv"
)
Ribo_deseqOut_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/Ribo/S/no_confounders/DE_table.tsv"
)

mRNA_metadata_df = pd.read_csv(
    mRNA_metadata_path, delimiter="\t", index_col=None, header=0
)
Ribo_metadata_df = pd.read_csv(
    Ribo_metadata_path, delimiter="\t", index_col=None, header=0
)

In [ ]:
# first, check the log2FC correspondence

mRNA_deseq_df = pd.read_csv(
    mRNA_deseqOut_path, delimiter="\t", index_col=None, header=0
)
Ribo_deseq_df = pd.read_csv(
    Ribo_deseqOut_path, delimiter="\t", index_col=None, header=0
)

cols = list(mRNA_deseq_df.columns)[1:]
rename_dict = {}
for elem in cols:
    rename_dict[elem] = elem + "_mRNA"
mRNA_deseq_df = mRNA_deseq_df.rename(columns=rename_dict)

cols = list(Ribo_deseq_df.columns)[1:]
rename_dict = {}
for elem in cols:
    rename_dict[elem] = elem + "_Ribo"
Ribo_deseq_df = Ribo_deseq_df.rename(columns=rename_dict)

correspondence_df = pd.merge(
    mRNA_deseq_df, Ribo_deseq_df, how="inner", on="gene_name"
).dropna()

In [ ]:
sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(5, 5))

data = correspondence_df.copy()

sign_thr = 0.05


def assign_category(x):
    if x["padj_mRNA"] < sign_thr:
        if x["padj_Ribo"] < sign_thr:
            return "both"
        else:
            return "only mRNA"
    else:
        if x["padj_Ribo"] < sign_thr:
            return "only Ribo"
        else:
            return "none"


data["category"] = data.apply(lambda x: assign_category(x), 1)
data["t"] = 1
gr = data.groupby("category").agg({"t": np.sum}).reset_index()
relabel_dict = {}
for index_gr, row_gr in gr.iterrows():
    relabel_dict[row_gr["category"]] = (
        row_gr["category"] + ", " + str(np.round(row_gr["t"] / 1000, 2)) + " K genes"
    )
data["DE significance"] = data["category"].map(relabel_dict).astype("category")
gr["DE significance"] = gr["category"].map(relabel_dict).astype("category")

# data['-$log_{10}padj$'] = (-1)*np.log10((data['padj_mRNA']*data['padj_Ribo'])**0.5)
data["-$log_{10}padj$"] = (-1) * np.log10(data[["padj_mRNA", "padj_Ribo"]].min(1))

x_feature, y_feature, hue = (
    "log2FoldChange_mRNA",
    "log2FoldChange_Ribo",
    "DE significance",
)
xlabel, ylabel = "$log_2FC$ mRNA", "$log_2FC$ Ribo"
hue_order = list(gr.sort_values(["t"], ascending=True)[hue])
palette = ["red", "orange", "blue", "grey"]
alpha_param, s_param = 0.3, 10
size_param = "-$log_{10}padj$"

ax = sns.scatterplot(
    data=data,
    x=x_feature,
    y=y_feature,
    hue=hue,
    hue_order=hue_order,
    palette=palette,
    alpha=alpha_param,
    size=size_param,
    sizes=(1, 40),
)
diag_df = pd.DataFrame(
    [[ax.get_xlim()[0], ax.get_xlim()[0]], [ax.get_xlim()[1], ax.get_xlim()[1]]],
    columns=[x_feature, y_feature],
)
ax = sns.lineplot(
    data=diag_df, y=y_feature, x=x_feature, color="black", label="diagonal"
)

# fit linear models
from statsmodels.regression.linear_model import OLS
from statsmodels.regression.linear_model import WLS
from statsmodels.regression.quantile_regression import QuantReg

data["c"] = 1  # constant
X = data[["c", x_feature]].values
y = data[[y_feature]].values

mod = OLS(endog=y, exog=X)
reg = mod.fit()
data["y_pred"] = reg.predict(X)
ax = sns.lineplot(data=data, y="y_pred", x=x_feature, color="green", label="OLS")

w = data["-$log_{10}padj$"].values
mod = WLS(endog=y, exog=X, weights=w)
reg = mod.fit()
data["y_pred"] = reg.predict(X)
ax = sns.lineplot(
    data=data, y="y_pred", x=x_feature, color="lime", label="WLS (weigthed by padj)"
)

mod = QuantReg(endog=y, exog=X)
reg = mod.fit(q=0.5, max_iter=10000)
data["y_pred"] = reg.predict(X)
ax = sns.lineplot(
    data=data, y="y_pred", x=x_feature, color="olive", label="quantile regression"
)

ax.tick_params(left=True, bottom=True)
ax.legend(
    bbox_to_anchor=(1.05, 1.00),
    loc="upper left",
    borderaxespad=0.0,
    title="",
    ncols=1,
    markerscale=1,
)

ax.set(xlabel=xlabel, ylabel=ylabel)

out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "gene_expression/genomic_mapping_UM/",
    shell=True,
)
fig.savefig(
    subdirs["figures_dir"]
    + "gene_expression/genomic_mapping_UM/log2FC_mRNA_vs_log2FC_Ribo.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"]
    + "gene_expression/genomic_mapping_UM/log2FC_mRNA_vs_log2FC_Ribo.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
out = subprocess.check_output(
    "mkdir -p "
    + subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis_TE/",
    shell=True,
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis_TE/new.TE_up.Ribo_up.tsv"
)
data.loc[(data["category"] == "only Ribo") & (data["log2FoldChange_Ribo"] > 0)][
    ["gene_name"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)
out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis_TE/new.TE_down.Ribo_down.tsv"
)
data.loc[(data["category"] == "only Ribo") & (data["log2FoldChange_Ribo"] < 0)][
    ["gene_name"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis_TE/new.TE_up.mRNA_down.tsv"
)
data.loc[(data["category"] == "only mRNA") & (data["log2FoldChange_mRNA"] < 0)][
    ["gene_name"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis_TE/new.TE_unchanged.mRNA_down.tsv"
)
data.loc[(data["category"] == "both") & (data["log2FoldChange_mRNA"] < 0)][
    ["gene_name"]
].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

out_file_path = (
    subdirs["tables_dir"]
    + "gene_expression/genomic_mapping_UM/enrichment_analysis_TE/all_genes.tsv"
)
data[["gene_name"]].dropna().drop_duplicates().to_csv(
    out_file_path, sep=str("\t"), header=False, index=None
)

In [ ]:
outdir_path = subdirs["tables_dir"] + "gene_expression/genomic_mapping_UM/TE/S/"

out = subprocess.check_output("mkdir -p " + outdir_path, shell=True)

# run differential analysis, accounting for replicates and SFs being confounders
# conda activate deseq

ommand = (
    command
    + """Rscript """
    + subdirs["wf_dir"]
    + "scripts/DTEG.r"
    + " "
    + Ribo_path
    + " "
    + mRNA_path
    + " "
)
command

In [ ]:
Ribo_metadata_df["SeqType"] = "RIBO"
mRNA_metadata_df["SeqType"] = "RNA"
sample_informx_feature, y_feature, hue = (
    "log2_baseMean",
    "log2FoldChange",
    "DE_labelled",
)
alpha_param, s_param = 0.3, 5
ation_df = pd.concat(
    [
        mRNA_metadata_df[["sample", "EXP_CTL", "SeqType"]],
        Ribo_metadata_df[["sample", "EXP_CTL", "SeqType"]],
    ]
).reset_index(drop=True)
sample_information_df["Condition"] = sample_information_df["EXP_CTL"].map(
    {"EXP": 2, "CTL": 1}
)
sample_information_df